# PIKA: Physics-Informed Koopman Autoencoder for Glacier Mass Balance Forecasting

**NeurIPS 2026 CCAI Workshop — "Fostering Ground-Up Innovation in AI and Climate"**

A small residual Koopman autoencoder (~6k parameters) for 5-year glacier mass-balance forecasting, compared to LSTM, tree models, per-glacier PDD/TI, and persistence on a **leakage-safe expanded protocol** (101 train / 20 assigned holdout; no Antarctic, Himalayan, Greenland, or NZ glacier in training).

**What holds:** residual/delta learning is load-bearing; PIKA 16/32 is statistically tied with a 9× larger LSTM on IID pooled RMSE (5 seeds, CI includes 0; LSTM’s mean is lower); conformal UQ is reported raw and calibrated.

**What does not:** physics and Lyapunov regularizers do not reduce RMSE; LSTM wins every IID horizon including h=1; 16/32 is not the pooled OOD winner (48/96 is); LORO does not show positive transfer.


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from scipy import stats
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Dict, List, Tuple
import warnings
import copy
import math

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.13.0


## 1. Configuration & Data Loading

In [2]:
# -- Constants --
HISTORY_LEN = 5
FORECAST_LEN = 5
CLIMATE_COLS = ['pdd', 'solid_precip_mm', 'summer_temp_c']
CLIMATE_DIM = len(CLIMATE_COLS)
SEED = 0

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -- Data paths --
# Kaggle first, then local: run from repo root or from final/.
_DATA_CANDIDATES = [
    Path('data'),
    Path('../data'),
    Path.cwd().parent / 'data',
]
DATA_DIR = next(
    (p for p in _DATA_CANDIDATES
     if (p / 'master_glacier_data_expanded.csv').exists()),
    Path('data'),
)
print(f"Data directory: {DATA_DIR}")

# -- Candidate routing --
DROP_IDS = [3454]                           # Dokriani — unvalidated provenance
EXTERNAL_HOLDOUT_IDS = [3366, 3367,         # Johnsons, Hurd (Antarctic RGI-19)
                        3996, 3997]         # Mera, Pokalde (Himalaya RGI-15)


def extract_rgi_region(rgi_id_series):
    """Extract integer region from RGI ID like 'RGI60-03.04539' -> 3."""
    return rgi_id_series.str.extract(r'RGI\d+-(\d+)\.', expand=False).astype(float).astype('Int64')


def load_data(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Load the leakage-safe expanded master when present; else the 76/10 snapshot."""
    expanded_path = data_dir / 'master_glacier_data_expanded.csv'
    if expanded_path.exists():
        df = pd.read_csv(expanded_path)
        print('Loaded master_glacier_data_expanded.csv')
        print('Roles already assigned: Himalaya/Antarctic/NZ/Greenland/new Andes = holdout.')
        print('Dokriani and Hamaguri Yuki already dropped.')
    else:
        master = pd.read_csv(data_dir / 'master_glacier_data_fixed.csv')
        candidates = pd.read_csv(data_dir / 'candidate_master_rows.csv')
        candidates = candidates[~candidates['glacier_id'].isin(DROP_IDS)].copy()
        print(f"Dropped Dokriani (id=3454) -- unvalidated, unknown provenance")
        candidates.loc[
            candidates['glacier_id'].isin(EXTERNAL_HOLDOUT_IDS), 'role'
        ] = 'external_holdout'
        df = pd.concat([master, candidates], ignore_index=True)

    # Derive rgi_region from rgi_id
    if 'rgi_region' not in df.columns and 'rgi_id' in df.columns:
        df['rgi_region'] = extract_rgi_region(df['rgi_id'])

    # Standardize elevation column name
    if 'mean_elevation' not in df.columns and 'elevation_med_m' in df.columns:
        df['mean_elevation'] = df['elevation_med_m']

    # Drop TRAINING glaciers with no RGI geometry (all static attrs null).
    # Do NOT impute: the physics module maps (elevation, area) -> DDF.
    # Holdouts without geometry are kept for mass-balance OOD tests;
    # physics is simply not applied to them.
    static_cols = [c for c in ['elevation_med_m', 'area_km2', 'slope_deg', 'aspect_deg']
                   if c in df.columns]
    if static_cols:
        train_ids_all = df.loc[df['role'] == 'training_population', 'glacier_id'].unique()
        dropped = []
        for gid in train_ids_all:
            g = df[df['glacier_id'] == gid]
            if g[static_cols].isna().all().all():
                dropped.append(gid)
        if dropped:
            names = (
                df.loc[df['glacier_id'].isin(dropped), ['glacier_id', 'glacier_name']]
                .drop_duplicates()
            )
            print(f"Dropped {len(dropped)} training glacier(s) with no RGI geometry:")
            for _, row in names.iterrows():
                print(f"  {int(row['glacier_id']):>5d}  {row['glacier_name']}")
            df = df[~df['glacier_id'].isin(dropped)].copy()

    # Merge WGMS uncertainty if file exists
    wgms_path = data_dir / 'wgms_mass_balance_slim.csv'
    if wgms_path.exists():
        wgms = pd.read_csv(wgms_path)
        if 'uncertainty' in wgms.columns and 'wgms_uncertainty' not in df.columns:
            wgms_unc = (
                wgms.groupby('glacier_id')['uncertainty']
                .mean()
                .reset_index()
                .rename(columns={'uncertainty': 'wgms_uncertainty'})
            )
            df = df.merge(wgms_unc, on='glacier_id', how='left')

    # Print summary
    train_mask = df['role'] == 'training_population'
    hold_mask = df['role'] == 'external_holdout'
    n_train = df.loc[train_mask, 'glacier_id'].nunique()
    n_hold = df.loc[hold_mask, 'glacier_id'].nunique()
    holdout_names = (
        df.loc[hold_mask, ['glacier_id', 'glacier_name']]
        .drop_duplicates()
        .sort_values('glacier_id')
    )

    if 'rgi_region' in df.columns:
        regions = sorted(df.loc[hold_mask, 'rgi_region'].dropna().unique())
    else:
        regions = []

    print(f"\n{'='*60}")
    print(f"Training glaciers : {n_train}")
    print(f"Holdout glaciers  : {n_hold}")
    print(f"Holdout regions   : {regions}")
    print(f"\nHoldout roster:")
    for _, row in holdout_names.iterrows():
        reg = df.loc[df['glacier_id'] == row['glacier_id'], 'rgi_region'].iloc[0] if 'rgi_region' in df.columns else '?'
        print(f"  {row['glacier_id']:>5d}  {row['glacier_name']}  (RGI-{reg})")
    print(f"{'='*60}")

    return df


df = load_data()


Data directory: data
Loaded master_glacier_data_expanded.csv
Roles already assigned: Himalaya/Antarctic/NZ/Greenland/new Andes = holdout.
Dokriani and Hamaguri Yuki already dropped.

Training glaciers : 101
Holdout glaciers  : 20
Holdout regions   : [np.int64(5), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]

Holdout roster:
    226  YANAMAREY  (RGI-16)
   1538  ROLLESTON  (RGI-18)
   1597  BREWSTER  (RGI-18)
   1624  ANTIZANA15ALPHA  (RGI-16)
   1629  MITTIVAKKAT  (RGI-5)
   2000  MARTIAL ESTE  (RGI-17)
   2665  BAHIA DEL DIABLO  (RGI-19)
   2667  CHARQUINI SUR  (RGI-16)
   2721  CONEJERAS  (RGI-16)
   2921  CHHOTA SHIGRI  (RGI-14)
   3350  FREYA  (RGI-5)
   3366  JOHNSONS  (RGI-19)
   3367  HURD  (RGI-19)
   3902  CONCONTA NORTE  (RGI-17)
   3903  BROWN SUPERIOR  (RGI-17)
   3972  MOCHO CHOSHUENCO SE  (RGI-17)
   3987  PARLUNG NO. 94  (RGI-15)
   3996  MERA  (RGI-15)
   3997  POKALDE  (RGI-15)
  35232  ARTESONRAJU  (RGI-16)


In [3]:
def build_sequences(
    df: pd.DataFrame,
    glacier_ids: list,
    history_len: int = HISTORY_LEN,
    forecast_len: int = FORECAST_LEN,
    max_estimated_years: int = 2,
) -> List[Dict]:
    """
    Sliding-window sequence builder.
    Windows with too many estimated years are skipped.
    """
    seqs = []
    total_len = history_len + forecast_len

    for gid in glacier_ids:
        gdf = df[df['glacier_id'] == gid].sort_values('year').reset_index(drop=True)
        if len(gdf) < total_len:
            continue

        has_estimated = 'window_estimated' in gdf.columns

        for start in range(len(gdf) - total_len + 1):
            window = gdf.iloc[start : start + total_len]

            if has_estimated:
                n_est = window['window_estimated'].sum()
                if n_est > max_estimated_years:
                    continue

            # Check for NaN in balance or climate
            if window['annual_balance'].isna().any():
                continue
            if window[CLIMATE_COLS].isna().any().any():
                continue

            hist = window.iloc[:history_len]
            fut = window.iloc[history_len:]

            elev_col = 'mean_elevation' if 'mean_elevation' in gdf.columns else 'elevation_med_m'
            area_col = 'area_km2'
            elev_val = hist[elev_col].iloc[0] if elev_col in hist.columns else np.nan
            area_val = hist[area_col].iloc[0] if area_col in hist.columns else np.nan
            # Training sequences require real geometry. Skip rather than invent.
            if not (pd.notna(elev_val) and pd.notna(area_val)):
                continue

            seq = {
                'glacier_id': gid,
                'hist_b': hist['annual_balance'].values.astype(np.float32),
                'hist_c': hist[CLIMATE_COLS].values.astype(np.float32),
                'fut_c': fut[CLIMATE_COLS].values.astype(np.float32),
                'target_b': fut['annual_balance'].values.astype(np.float32),
                'elevation': float(elev_val),
                'area': float(area_val),
                'lat': float(hist['lat'].iloc[0]) if 'lat' in hist.columns and pd.notna(hist['lat'].iloc[0]) else 0.0,
                'hist_years': hist['year'].values.tolist(),
                'fut_years': fut['year'].values.tolist(),
            }
            seqs.append(seq)

    return seqs


def build_test_window(
    df: pd.DataFrame,
    glacier_ids: list,
    history_len: int = HISTORY_LEN,
    forecast_len: int = FORECAST_LEN,
) -> List[Dict]:
    """
    Strict test windows: only the LAST valid window per glacier,
    with max_estimated_years=0 and no NaN.
    """
    seqs = []
    total_len = history_len + forecast_len

    for gid in glacier_ids:
        gdf = df[df['glacier_id'] == gid].sort_values('year').reset_index(drop=True)
        if len(gdf) < total_len:
            continue

        has_estimated = 'window_estimated' in gdf.columns

        for start in range(len(gdf) - total_len, -1, -1):
            window = gdf.iloc[start : start + total_len]

            if has_estimated and window['window_estimated'].sum() > 0:
                continue
            if window['annual_balance'].isna().any():
                continue
            if window[CLIMATE_COLS].isna().any().any():
                continue

            hist = window.iloc[:history_len]
            fut = window.iloc[history_len:]

            elev_col = 'mean_elevation' if 'mean_elevation' in gdf.columns else 'elevation_med_m'
            area_col = 'area_km2'

            seq = {
                'glacier_id': gid,
                'hist_b': hist['annual_balance'].values.astype(np.float32),
                'hist_c': hist[CLIMATE_COLS].values.astype(np.float32),
                'fut_c': fut[CLIMATE_COLS].values.astype(np.float32),
                'target_b': fut['annual_balance'].values.astype(np.float32),
                'elevation': float(hist[elev_col].iloc[0]) if (elev_col in hist.columns and pd.notna(hist[elev_col].iloc[0])) else float('nan'),
                'area': float(hist[area_col].iloc[0]) if (area_col in hist.columns and pd.notna(hist[area_col].iloc[0])) else float('nan'),
                'lat': float(hist['lat'].iloc[0]) if 'lat' in hist.columns and pd.notna(hist['lat'].iloc[0]) else 0.0,
                'hist_years': hist['year'].values.tolist(),
                'fut_years': fut['year'].values.tolist(),
            }
            seqs.append(seq)
            break  # only last valid window

    return seqs


# Quick test
train_ids = df.loc[df['role'] == 'training_population', 'glacier_id'].unique().tolist()
holdout_ids = df.loc[df['role'] == 'external_holdout', 'glacier_id'].unique().tolist()
print(f"Train glaciers: {len(train_ids)}, Holdout glaciers: {len(holdout_ids)}")
print(f"Columns available: {sorted(df.columns.tolist())}")


Train glaciers: 101, Holdout glaciers: 20
Columns available: ['annual_balance', 'annual_balance_unc', 'area_km2', 'aspect_deg', 'begin_date', 'elevation_med_m', 'end_date', 'glacier_id', 'glacier_name', 'lat', 'lon', 'mean_elevation', 'n_days_in_window', 'pdd', 'rain_mm', 'rgi_id', 'rgi_region', 'role', 'slope_deg', 'solid_precip_mm', 'solid_precip_v2', 'summer_temp_c', 'total_precip_mm', 'window_estimated', 'year']


In [4]:
def seqs_to_tensors(
    seqs: List[Dict],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Stack sequence dicts into batched tensors."""
    hist_b = torch.tensor(
        np.stack([s['hist_b'] for s in seqs]), dtype=torch.float32
    )
    hist_c = torch.tensor(
        np.stack([s['hist_c'] for s in seqs]), dtype=torch.float32
    )
    fut_c = torch.tensor(
        np.stack([s['fut_c'] for s in seqs]), dtype=torch.float32
    )
    target_b = torch.tensor(
        np.stack([s['target_b'] for s in seqs]), dtype=torch.float32
    )
    return hist_b, hist_c, fut_c, target_b


class Scaler:
    """Z-score normalization.

    scalar=True  -> one mean/std over the whole tensor (use for mass balance).
    scalar=False -> mean/std over dim 0, keeping feature axis (use for climate).
    """

    def __init__(self, data: torch.Tensor, scalar: bool = True, eps: float = 1e-8):
        if scalar:
            self.mean = data.mean()
            self.std = data.std().clamp(min=eps)
        else:
            self.mean = data.mean(dim=0)
            self.std = data.std(dim=0).clamp(min=eps)

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.mean) / self.std

    def inverse(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.std + self.mean

    def to(self, device: torch.device) -> 'Scaler':
        self.mean = self.mean.to(device)
        self.std = self.std.to(device)
        return self


print("Scaler and tensor utilities ready.")

Scaler and tensor utilities ready.


## 2. Model Architecture: PIKA

**Design (stable version):**

1. **Climate-conditioned encoder** — Encodes historical balance *and* climate jointly, so the latent state captures regime information.
2. **Continuous-time ODO Koopman operator** — Orthogonal–Diagonal–Orthogonal factorization; orthogonals via **QR** of a free matrix (Cayley \(I+S\) went singular). Matrix exponential gives \(K = e^{A\Delta t}\). Real eigenvalues only, spectral radius buffered at \(\rho_{\max}=0.95\).
3. **Additive climate control** — An MLP maps future climate into a latent increment at each forecast step: \(z \leftarrow Kz + \mathrm{control}(c)\). Not bilinear.
4. **Residual / delta learning** — The decoder predicts the change from the last observed balance. Ablation: removing this is the only large RMSE hit.
5. **Differentiable temperature-index prior** — Geometry-conditioned DDF used as a **regularizer only** (not added into the forecast). Ablation: does not improve RMSE.
6. **Monotone quantile decoder** — Auxiliary pinball head; the RMSE point forecast is the MSE head. Intervals are shown raw and after glacier-level conformal calibration.


In [5]:
class OrthogonalLayer(nn.Module):
    """
    Orthogonal matrix via QR factorization of a free matrix.
    More numerically stable than Cayley (I+S)^{-1}, which can become singular.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        # Initialize near identity so early training is well-conditioned
        W = torch.eye(dim) + torch.randn(dim, dim) * 0.01
        self.weight = nn.Parameter(W)

    def forward(self) -> torch.Tensor:
        Q, R = torch.linalg.qr(self.weight)
        # Make the factorization unique: absorb sign of R diagonal into Q
        diag_sign = torch.sign(torch.diag(R)).clamp(min=1.0)  # +1 if zero
        return Q * diag_sign.unsqueeze(0)


print("OrthogonalLayer ready (QR parameterization).")
V = OrthogonalLayer(8)
mat = V()
orth_check = torch.allclose(mat @ mat.T, torch.eye(8), atol=1e-5)
print(f"  Orthogonality check (dim=8): {orth_check}")
print(f"  det(V)={torch.det(mat).item():.4f}")


OrthogonalLayer ready (QR parameterization).
  Orthogonality check (dim=8): True
  det(V)=-1.0000


In [6]:
class DifferentiableTemperatureIndex(nn.Module):
    """
    Differentiable temperature-index physics module.
    Maps (elevation, area) -> glacier-specific degree-day factor.
    Used as a TRAINING regularizer only, not as an additive backbone at inference.
    """

    def __init__(self, hidden: int = 32):
        super().__init__()
        self.ddf_net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 2),
        )
        self.register_buffer('elev_mean', torch.tensor(0.0))
        self.register_buffer('elev_std', torch.tensor(1.0))
        self.register_buffer('area_mean', torch.tensor(0.0))
        self.register_buffer('area_std', torch.tensor(1.0))

    def set_normalization(self, elev_mean, elev_std, area_mean, area_std):
        self.elev_mean.fill_(float(elev_mean))
        self.elev_std.fill_(max(float(elev_std), 1e-6))
        self.area_mean.fill_(float(area_mean))
        self.area_std.fill_(max(float(area_std), 1e-6))

    def forward(self, climate_norm, elevation, area):
        elev_n = (elevation - self.elev_mean) / self.elev_std
        area_n = (area - self.area_mean) / self.area_std
        geo = torch.stack([elev_n, area_n], dim=-1)
        params = self.ddf_net(geo)
        ddf = F.softplus(params[:, 0])
        alpha = F.softplus(params[:, 1])
        pdd = climate_norm[:, :, 0]
        precip = climate_norm[:, :, 1]
        return -ddf.unsqueeze(1) * pdd + alpha.unsqueeze(1) * precip


print("DifferentiableTemperatureIndex ready.")


DifferentiableTemperatureIndex ready.


In [7]:
class QuantileDecoder(nn.Module):
    """
    Per-step monotone quantile head.
    Input  z: (batch, latent_dim)
    Output : (batch, N_QUANTILES) with q0.1 <= ... <= q0.9
    """

    QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]
    N_QUANTILES = len(QUANTILES)

    def __init__(self, latent_dim: int, hidden: int, forecast_len: int = FORECAST_LEN):
        super().__init__()
        self.forecast_len = forecast_len  # unused; kept for call-site compatibility
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, self.N_QUANTILES),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        raw = self.net(z)  # (batch, n_q)
        median_idx = self.N_QUANTILES // 2
        median = raw[:, median_idx:median_idx + 1]
        lower_gaps = F.softplus(raw[:, :median_idx])
        upper_gaps = F.softplus(raw[:, median_idx + 1:])
        lower = median - torch.cumsum(lower_gaps.flip(-1), dim=-1).flip(-1)
        upper = median + torch.cumsum(upper_gaps, dim=-1)
        return torch.cat([lower, median, upper], dim=-1)


print("QuantileDecoder ready (per-step).")


QuantileDecoder ready (per-step).


In [8]:
class PIKAv2(nn.Module):
    """
    Physics-Informed Koopman Autoencoder v2 (stable version).

    encoder  : concat(hist_b, flatten(hist_c)) -> z0
    Koopman  : A = V diag(lambda) V^T (ODO); K = exp(A dt)
    control  : MLP(climate) added each step
    unroll   : z_{t+1} = z_t @ K^T + control(c_t) ; decode EVERY z_t
    decoder  : residual/delta in normalized balance space
    physics  : training regularizer only (not added into the prediction)
    """

    def __init__(
        self,
        latent_dim: int = 48,
        hidden: int = 96,
        history_len: int = HISTORY_LEN,
        forecast_len: int = FORECAST_LEN,
        climate_dim: int = CLIMATE_DIM,
        use_quantiles: bool = True,
        use_physics: bool = True,
        dt: float = 1.0,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden = hidden
        self.history_len = history_len
        self.forecast_len = forecast_len
        self.climate_dim = climate_dim
        self.use_quantiles = use_quantiles
        self.use_physics = use_physics
        self.dt = dt

        enc_input_dim = history_len + history_len * climate_dim
        self.encoder = nn.Sequential(
            nn.Linear(enc_input_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Linear(hidden, latent_dim),
            nn.LayerNorm(latent_dim),
        )

        self.orth = OrthogonalLayer(latent_dim)
        self.log_eigs_raw = nn.Parameter(torch.randn(latent_dim) * 0.1)
        self.register_buffer('rho_max', torch.tensor(0.95))

        self.climate_control = nn.Sequential(
            nn.Linear(climate_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, latent_dim),
        )

        # MSE point head — this is what we report as y_pred (LSTM is trained on MSE).
        # Pinball median is biased (q0.5 covered ~72% of obs); do not use it for RMSE.
        self.point_decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),
        )
        # Bounded per-step climate skip (tanh so OOD climate cannot explode).
        self.climate_skip = nn.Linear(climate_dim, 1)
        nn.init.zeros_(self.climate_skip.weight)
        nn.init.zeros_(self.climate_skip.bias)
        if use_quantiles:
            self.quantile_decoder = QuantileDecoder(latent_dim, hidden, forecast_len)

        if use_physics:
            self.physics = DifferentiableTemperatureIndex(hidden=32)

    def _eigenvalues(self) -> torch.Tensor:
        eig_min = -0.7
        eig_max = math.log(self.rho_max.item())
        return eig_min + (eig_max - eig_min) * torch.sigmoid(self.log_eigs_raw)

    def _build_koopman(self) -> torch.Tensor:
        V = self.orth()
        eigs = self._eigenvalues()
        A = V @ torch.diag(eigs) @ V.T
        return torch.matrix_exp(A * self.dt)

    def forward(self, hist_b, hist_c, fut_c, elevation=None, area=None, lat=None, residual_shrink=None):
        # lat / residual_shrink ignored — kept so old call sites do not crash
        B = hist_b.shape[0]
        z = self.encoder(torch.cat([hist_b, hist_c.reshape(B, -1)], dim=-1))
        K = self._build_koopman()

        q_steps, d_steps = [], []
        for t in range(self.forecast_len):
            c_t = fut_c[:, t, :]
            z = z @ K.T + self.climate_control(c_t)
            point_t = self.point_decoder(z).squeeze(-1) + torch.tanh(self.climate_skip(c_t)).squeeze(-1)
            d_steps.append(point_t)
            if self.use_quantiles:
                q_raw = self.quantile_decoder(z)
                med = q_raw[:, QuantileDecoder.N_QUANTILES // 2 : QuantileDecoder.N_QUANTILES // 2 + 1]
                q_t = q_raw - med + point_t.unsqueeze(-1)
                q_steps.append(q_t)

        out = {'delta': torch.stack(d_steps, dim=1)}
        if self.use_quantiles:
            out['quantiles'] = torch.stack(q_steps, dim=1)
        if self.use_physics and elevation is not None and area is not None:
            finite = torch.isfinite(elevation) & torch.isfinite(area)
            if finite.any():
                out['physics'] = self.physics(fut_c, torch.nan_to_num(elevation, 0.0),
                                              torch.nan_to_num(area, 0.0))
        return out

    def lyapunov_loss(self) -> torch.Tensor:
        exp_eigs = torch.exp(self._eigenvalues() * self.dt)
        return F.relu(0.5 - exp_eigs.abs().max())

    def spectral_info(self) -> Dict:
        with torch.no_grad():
            exp_eigs = torch.exp(self._eigenvalues() * self.dt)
            mags = exp_eigs.abs()
            return {
                'spectral_radius': mags.max().item(),
                'eig_min': mags.min().item(),
                'eig_max': mags.max().item(),
                'eig_mean': mags.mean().item(),
                'eigenvalues': mags.cpu().numpy().tolist(),
            }


_model = PIKAv2(latent_dim=16, hidden=32).to(device)
print(f"PIKAv2 parameter count: {sum(p.numel() for p in _model.parameters()):,}")
_hb = torch.randn(4, HISTORY_LEN, device=device)
_hc = torch.randn(4, HISTORY_LEN, CLIMATE_DIM, device=device)
_fc = torch.randn(4, FORECAST_LEN, CLIMATE_DIM, device=device)
_out = _model(_hb, _hc, _fc)
print(f"Output delta shape : {_out['delta'].shape}")
print(f"Output quantiles   : {_out['quantiles'].shape}")
print(f"Spectral radius    : {_model.spectral_info()['spectral_radius']:.3f}")
del _model, _hb, _hc, _fc, _out


PIKAv2 parameter count: 5,852


Output delta shape : torch.Size([4, 5])
Output quantiles   : torch.Size([4, 5, 5])
Spectral radius    : 0.707


In [9]:
# ══════════════════════════════════════════════════════════════════════
#  BASELINE MODELS
# ══════════════════════════════════════════════════════════════════════


class LSTMForecaster(nn.Module):
    """LSTM baseline with autoregressive decoding."""

    def __init__(
        self,
        input_size: int = 1 + CLIMATE_DIM,
        hidden_size: int = 64,
        forecast_len: int = FORECAST_LEN,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.forecast_len = forecast_len
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(
        self,
        hist_b: torch.Tensor,
        hist_c: torch.Tensor,
        fut_c: torch.Tensor,
    ) -> torch.Tensor:
        B = hist_b.shape[0]
        x = torch.cat([hist_b.unsqueeze(-1), hist_c], dim=-1)
        _, (h, c) = self.lstm(x)

        preds = []
        last_b = hist_b[:, -1:]
        for t in range(self.forecast_len):
            inp = torch.cat([last_b.unsqueeze(1), fut_c[:, t:t+1, :]], dim=-1)
            out, (h, c) = self.lstm(inp, (h, c))
            pred = self.fc(out.squeeze(1))
            preds.append(pred)
            last_b = pred

        return torch.cat(preds, dim=-1)


class PIKAv2_Reduced(PIKAv2):
    """Reduced-capacity PIKAv2 for complexity comparison."""

    def __init__(self, **kwargs):
        defaults = dict(latent_dim=16, hidden=32, use_quantiles=False, use_physics=False)
        defaults.update(kwargs)
        super().__init__(**defaults)


def train_xgboost_baseline(
    train_seqs: List[Dict],
    forecast_len: int = FORECAST_LEN,
) -> List[GradientBoostingRegressor]:
    """Fit one GradientBoosting model per forecast horizon step."""
    X_list, Y_list = [], []
    for s in train_seqs:
        feat = np.concatenate([s['hist_b'], s['hist_c'].ravel(), s['fut_c'].ravel()])
        X_list.append(feat)
        Y_list.append(s['target_b'])

    X = np.stack(X_list)
    Y = np.stack(Y_list)

    models = []
    for h in range(forecast_len):
        gbr = GradientBoostingRegressor(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, random_state=SEED,
        )
        gbr.fit(X, Y[:, h])
        models.append(gbr)

    return models


def predict_xgboost(
    models: List[GradientBoostingRegressor],
    seqs: List[Dict],
) -> np.ndarray:
    X = np.stack([
        np.concatenate([s['hist_b'], s['hist_c'].ravel(), s['fut_c'].ravel()])
        for s in seqs
    ])
    return np.column_stack([m.predict(X) for m in models])


def train_rf_baseline(
    train_seqs: List[Dict],
    forecast_len: int = FORECAST_LEN,
) -> List[RandomForestRegressor]:
    """Fit one RandomForest model per forecast horizon step."""
    X_list, Y_list = [], []
    for s in train_seqs:
        feat = np.concatenate([s['hist_b'], s['hist_c'].ravel(), s['fut_c'].ravel()])
        X_list.append(feat)
        Y_list.append(s['target_b'])

    X = np.stack(X_list)
    Y = np.stack(Y_list)

    models = []
    for h in range(forecast_len):
        rf = RandomForestRegressor(
            n_estimators=200, max_depth=8, random_state=SEED,
        )
        rf.fit(X, Y[:, h])
        models.append(rf)

    return models


def predict_rf(
    models: List[RandomForestRegressor],
    seqs: List[Dict],
) -> np.ndarray:
    X = np.stack([
        np.concatenate([s['hist_b'], s['hist_c'].ravel(), s['fut_c'].ravel()])
        for s in seqs
    ])
    return np.column_stack([m.predict(X) for m in models])


def pdd_ti_baseline(
    train_seqs: List[Dict],
    test_seqs: List[Dict],
    forecast_len: int = FORECAST_LEN,
) -> np.ndarray:
    """
    Simple temperature-index: per-glacier linear regression
    balance ~ PDD + solid_precip, predict using future climate.
    """
    from collections import defaultdict
    glacier_data = defaultdict(lambda: {'X': [], 'y': []})

    for s in train_seqs:
        gid = s['glacier_id']
        all_c = np.vstack([s['hist_c'], s['fut_c']])
        all_b = np.concatenate([s['hist_b'], s['target_b']])
        for i in range(len(all_b)):
            glacier_data[gid]['X'].append(all_c[i, :2])
            glacier_data[gid]['y'].append(all_b[i])

    glacier_models = {}
    fallback = LinearRegression()
    all_X = np.vstack([np.array(v['X']) for v in glacier_data.values()])
    all_y = np.concatenate([np.array(v['y']) for v in glacier_data.values()])
    fallback.fit(all_X, all_y)

    for gid, data in glacier_data.items():
        X_g = np.array(data['X'])
        y_g = np.array(data['y'])
        if len(y_g) >= 5:
            lr = LinearRegression()
            lr.fit(X_g, y_g)
            glacier_models[gid] = lr

    preds = []
    for s in test_seqs:
        gid = s['glacier_id']
        model = glacier_models.get(gid, fallback)
        fut_feats = s['fut_c'][:, :2]
        pred = model.predict(fut_feats)
        preds.append(pred[:forecast_len])

    return np.stack(preds)


def naive_persistence(
    seqs: List[Dict],
    forecast_len: int = FORECAST_LEN,
) -> np.ndarray:
    """Repeat last known balance for all forecast steps."""
    return np.stack([
        np.full(forecast_len, s['hist_b'][-1]) for s in seqs
    ])


print("All 6 baselines defined:")
print("  1. LSTMForecaster")
print("  2. PIKAv2_Reduced (latent=16, hidden=32)")
print("  3. XGBoost (GradientBoosting per horizon)")
print("  4. Random Forest (per horizon)")
print("  5. PDD Temperature-Index (per-glacier linear)")
print("  6. Naive Persistence")

All 6 baselines defined:
  1. LSTMForecaster
  2. PIKAv2_Reduced (latent=16, hidden=32)
  3. XGBoost (GradientBoosting per horizon)
  4. Random Forest (per horizon)
  5. PDD Temperature-Index (per-glacier linear)
  6. Naive Persistence


## 3. Training Functions

In [10]:
@dataclass
class TrainConfig:
    epochs: int = 1200
    lr: float = 1e-3
    weight_decay: float = 1e-4
    physics_weight: float = 0.02
    lyapunov_weight: float = 0.01
    horizon_discount: float = 0.95
    latent_dim: int = 16
    hidden: int = 32
    use_quantiles: bool = True
    use_physics: bool = True
    # False = train directly on absolute balance instead of the delta from the
    # last observation. Used only by the residual ablation, which must RETRAIN:
    # taking a delta-trained model and reading its output as an absolute balance
    # measures a systematic offset, not the value of residual anchoring.
    use_residual: bool = True
    seed: int = 0


print(TrainConfig())


TrainConfig(epochs=1200, lr=0.001, weight_decay=0.0001, physics_weight=0.02, lyapunov_weight=0.01, horizon_discount=0.95, latent_dim=16, hidden=32, use_quantiles=True, use_physics=True, use_residual=True, seed=0)


In [11]:
def quantile_loss(
    pred_quantiles: torch.Tensor,
    target: torch.Tensor,
    quantiles: List[float] = QuantileDecoder.QUANTILES,
    sample_weight: torch.Tensor = None,
) -> torch.Tensor:
    """Pinball loss averaged over quantiles, steps, and batch."""
    target_expanded = target.unsqueeze(-1)
    errors = target_expanded - pred_quantiles
    q_tensor = torch.tensor(quantiles, device=pred_quantiles.device).view(1, 1, -1)
    loss = torch.max(q_tensor * errors, (q_tensor - 1) * errors)  # (B, T, Q)
    per_seq = loss.mean(dim=(-1, -2))  # (B,)
    if sample_weight is not None:
        w = sample_weight.reshape(-1)
        return (per_seq * w).mean()
    return per_seq.mean()


def crps_from_quantiles(pred_quantiles, y_true, quantiles=QuantileDecoder.QUANTILES):
    """Approximate CRPS from quantile predictions."""
    n_q = len(quantiles)
    crps_vals = []
    for i in range(len(y_true)):
        for t in range(y_true.shape[1] if y_true.ndim > 1 else 1):
            yt = y_true[i, t] if y_true.ndim > 1 else y_true[i]
            pq = pred_quantiles[i, t, :] if pred_quantiles.ndim > 2 else pred_quantiles[i, :]
            score = 0.0
            for q_idx, tau in enumerate(quantiles):
                e = yt - pq[q_idx]
                score += (tau - (1.0 if e < 0 else 0.0)) * e
            crps_vals.append(score * 2.0 / n_q)
    return np.mean(crps_vals)


def calibration_score(pred_quantiles, y_true, quantiles=QuantileDecoder.QUANTILES):
    """Compute empirical coverage for each quantile level."""
    coverages = {}
    for q_idx, tau in enumerate(quantiles):
        if pred_quantiles.ndim == 3:
            below = (y_true <= pred_quantiles[:, :, q_idx]).float().mean().item()
        else:
            below = (y_true <= pred_quantiles[:, q_idx]).float().mean().item()
        coverages[tau] = below
    return coverages


def train_pikav2(
    train_seqs: List[Dict],
    cfg: TrainConfig = TrainConfig(),
    verbose: bool = True,
) -> Tuple[PIKAv2, Tuple['Scaler', 'Scaler'], List[Dict]]:
    """Full training loop for PIKAv2."""
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)

    if cfg.use_physics:
        n_before = len(train_seqs)
        train_seqs = [
            s for s in train_seqs
            if np.isfinite(s['elevation']) and np.isfinite(s['area'])
        ]
        n_drop = n_before - len(train_seqs)
        if n_drop:
            print(f"  Dropping {n_drop} sequences with missing elevation/area (no imputation)")
        if not train_seqs:
            raise ValueError("No training sequences remain after dropping missing geometry")

    hist_b, hist_c, fut_c, target_b = seqs_to_tensors(train_seqs)

    b_scaler = Scaler(hist_b, scalar=True)
    c_all = torch.cat([hist_c, fut_c], dim=1)
    c_scaler = Scaler(c_all.reshape(-1, CLIMATE_DIM), scalar=False)

    hist_b_n = b_scaler.transform(hist_b).to(device)
    hist_c_n = c_scaler.transform(hist_c.reshape(-1, CLIMATE_DIM)).reshape(hist_c.shape).to(device)
    fut_c_n = c_scaler.transform(fut_c.reshape(-1, CLIMATE_DIM)).reshape(fut_c.shape).to(device)
    target_b_n = b_scaler.transform(target_b).to(device)

    # Safety: zero out any remaining NaN from data gaps
    hist_b_n = torch.nan_to_num(hist_b_n, 0.0)
    hist_c_n = torch.nan_to_num(hist_c_n, 0.0)
    fut_c_n = torch.nan_to_num(fut_c_n, 0.0)
    target_b_n = torch.nan_to_num(target_b_n, 0.0)

    last_b_n = hist_b_n[:, -1:].expand_as(target_b_n)
    target_delta_n = target_b_n - last_b_n
    # The model head is the same either way; only what it is asked to predict
    # changes. With use_residual=False the anchor is never added, so the network
    # must learn the glacier-specific offset itself.
    target_train_n = target_delta_n if cfg.use_residual else target_b_n

    b_scaler.to(device)
    c_scaler.to(device)

    model = PIKAv2(
        latent_dim=cfg.latent_dim,
        hidden=cfg.hidden,
        use_quantiles=cfg.use_quantiles,
        use_physics=cfg.use_physics,
    ).to(device)

    elev_t, area_t = None, None
    if cfg.use_physics:
        elevations = np.array([s['elevation'] for s in train_seqs], dtype=np.float32)
        areas = np.array([s['area'] for s in train_seqs], dtype=np.float32)
        model.physics.set_normalization(
            float(elevations.mean()), float(max(elevations.std(), 1e-6)),
            float(areas.mean()), float(max(areas.std(), 1e-6)),
        )
        elev_t = torch.tensor(elevations, device=device)
        area_t = torch.tensor(areas, device=device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    history = []
    horizon_weights = torch.tensor(
        [cfg.horizon_discount ** i for i in range(FORECAST_LEN)],
        device=device
    ).unsqueeze(0)

    for epoch in range(cfg.epochs):
        model.train()
        optimizer.zero_grad()

        out = model(hist_b_n, hist_c_n, fut_c_n, elev_t, area_t)

        # Main loss = MSE on the point head (same objective as LSTM / XGBoost).
        # Quantile pinball is auxiliary so UQ does not bias the RMSE forecast.
        delta_pred = out['delta']
        main_loss = (((delta_pred - target_train_n) ** 2) * horizon_weights).mean()
        if cfg.use_quantiles and 'quantiles' in out:
            main_loss = main_loss + 0.3 * quantile_loss(out['quantiles'], target_train_n)

        # Physics regularizes the MODEL toward the TI estimate (not TI toward data)
        physics_loss = torch.tensor(0.0, device=device)
        warmup_frac = min(1.0, epoch / (cfg.epochs * 0.1)) if cfg.use_physics else 0.0
        if cfg.use_physics and 'physics' in out:
            physics_est = out['physics']  # (B, forecast) in normalized climate units
            median_delta = out['delta']
            physics_loss_raw = ((median_delta - physics_est) ** 2).mean()
            physics_loss = physics_loss_raw * warmup_frac if torch.isfinite(physics_loss_raw) else torch.tensor(0.0, device=device)

        # Lyapunov loss
        lyap_loss = model.lyapunov_loss()

        total_loss = (
            main_loss
            + cfg.physics_weight * physics_loss
            + cfg.lyapunov_weight * lyap_loss
        )

        if not torch.isfinite(total_loss):
            # Skip this step entirely — don't corrupt model with NaN gradients
            optimizer.zero_grad()
            if verbose and epoch < 5:
                print(f"    [WARN] epoch {epoch}: total_loss=NaN, skipping")
            scheduler.step()
        else:
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

        # Logging
        with torch.no_grad():
            mse = ((out['delta'] - target_train_n) ** 2).mean().item()

        rec = {
            'epoch': epoch, 'loss': total_loss.item(), 'mse': mse,
            'physics': physics_loss.item(), 'lyapunov': lyap_loss.item(),
        }
        rec.update(model.spectral_info())
        history.append(rec)

        if verbose and (epoch % 200 == 0 or epoch == cfg.epochs - 1):
            print(f"  Epoch {epoch:4d} | loss={rec['loss']:.4f} "
                  f"mse={mse:.4f} phys={rec['physics']:.4f} "
                  f"rho={rec['spectral_radius']:.3f}")

    return model, (b_scaler, c_scaler), history


def train_lstm_baseline(
    train_seqs: List[Dict],
    epochs: int = 900,
    hidden: int = 64,
    seed: int = 0,
    verbose: bool = True,
) -> Tuple[LSTMForecaster, Tuple['Scaler', 'Scaler'], List]:
    """Train LSTM baseline (MSE loss, autoregressive)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    hist_b, hist_c, fut_c, target_b = seqs_to_tensors(train_seqs)

    b_scaler = Scaler(hist_b, scalar=True)
    c_all = torch.cat([hist_c, fut_c], dim=1)
    c_scaler = Scaler(c_all.reshape(-1, CLIMATE_DIM), scalar=False)

    hist_b_n = b_scaler.transform(hist_b).to(device)
    hist_c_n = c_scaler.transform(hist_c.reshape(-1, CLIMATE_DIM)).reshape(hist_c.shape).to(device)
    fut_c_n = c_scaler.transform(fut_c.reshape(-1, CLIMATE_DIM)).reshape(fut_c.shape).to(device)
    target_b_n = b_scaler.transform(target_b).to(device)

    # Safety: zero out any remaining NaN from data gaps
    hist_b_n = torch.nan_to_num(hist_b_n, 0.0)
    hist_c_n = torch.nan_to_num(hist_c_n, 0.0)
    fut_c_n = torch.nan_to_num(fut_c_n, 0.0)
    target_b_n = torch.nan_to_num(target_b_n, 0.0)

    b_scaler.to(device)
    c_scaler.to(device)

    model = LSTMForecaster(input_size=1 + CLIMATE_DIM, hidden_size=hidden).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(hist_b_n, hist_c_n, fut_c_n)
        loss = F.mse_loss(pred, target_b_n)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        history.append({'epoch': epoch, 'loss': loss.item()})

        if verbose and (epoch % 200 == 0 or epoch == epochs - 1):
            print(f"  Epoch {epoch:4d} | loss={loss.item():.4f}")

    return model, (b_scaler, c_scaler), history


## 4. Prediction & Evaluation Infrastructure

In [12]:
# ============================================================================
# Prediction helpers for every model family
# ============================================================================

def predict_pikav2(model, scalers, test_seqs, use_residual=True):
    """Generate predictions from PIKA.
    Model outputs DELTAS in normalised space; add back last_balance for absolute.
    use_residual=False matches a model trained with TrainConfig(use_residual=False),
    whose head already emits an absolute balance; must agree with training.
    test_seqs: list of dicts with keys hist_b, hist_c, fut_c, target_b, glacier_id
    Returns {gid: {y_pred, y_true, quantiles, anchor_balance}}
    """
    b_scaler, c_scaler = scalers
    model.eval()
    results = {}
    for seq in test_seqs:
        gid = seq['glacier_id']
        hist_b = torch.tensor(seq['hist_b'], dtype=torch.float32, device=device).unsqueeze(0)
        hist_c = torch.tensor(seq['hist_c'], dtype=torch.float32, device=device).unsqueeze(0)
        fut_c = torch.tensor(seq['fut_c'], dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            hist_b_n = b_scaler.transform(hist_b)
            hist_c_n = c_scaler.transform(hist_c.reshape(-1, CLIMATE_DIM)).reshape(hist_c.shape)
            fut_c_n = c_scaler.transform(fut_c.reshape(-1, CLIMATE_DIM)).reshape(fut_c.shape)
            out = model(hist_b_n, hist_c_n, fut_c_n)
            last_b_n = hist_b_n[:, -1:]

            delta_n = out['delta']
            abs_n = last_b_n + delta_n if use_residual else delta_n
            median_pred = b_scaler.inverse(abs_n[0]).detach().cpu().numpy().reshape(-1)
            quantile_preds = None
            if model.use_quantiles and 'quantiles' in out:
                abs_q_n = out['quantiles']
                if use_residual:
                    abs_q_n = last_b_n.unsqueeze(-1) + abs_q_n
                quantile_preds = b_scaler.inverse(abs_q_n[0]).detach().cpu().numpy()

        results[gid] = {
            'y_pred': median_pred,
            'y_true': seq['target_b'],
            'quantiles': quantile_preds,
            'anchor_balance': float(seq['hist_b'][-1]),
        }
    model.train()
    return results


def predict_lstm(model, scalers, test_seqs):
    """Generate predictions from LSTM."""
    b_scaler, c_scaler = scalers
    model.eval()
    results = {}
    for seq in test_seqs:
        gid = seq['glacier_id']
        hist_b = torch.tensor(seq['hist_b'], dtype=torch.float32, device=device).unsqueeze(0)
        hist_c = torch.tensor(seq['hist_c'], dtype=torch.float32, device=device).unsqueeze(0)
        fut_c = torch.tensor(seq['fut_c'], dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            hist_b_n = b_scaler.transform(hist_b)
            hist_c_n = c_scaler.transform(hist_c.reshape(-1, CLIMATE_DIM)).reshape(hist_c.shape)
            fut_c_n = c_scaler.transform(fut_c.reshape(-1, CLIMATE_DIM)).reshape(fut_c.shape)
            pred_n = model(hist_b_n, hist_c_n, fut_c_n)
            pred = b_scaler.inverse(pred_n[0]).detach().cpu().numpy().reshape(-1)
        results[gid] = {
            'y_pred': pred,
            'y_true': seq['target_b'],
            'anchor_balance': float(seq['hist_b'][-1]),
        }
    model.train()
    return results


def predict_sklearn(models, scalers, test_seqs):
    """Predict from per-horizon sklearn models (XGBoost or RF)."""
    b_scaler, c_scaler = scalers
    results = {}
    for seq in test_seqs:
        gid = seq['glacier_id']
        hist_b = torch.tensor(seq['hist_b'], dtype=torch.float32).unsqueeze(0)
        hist_c = torch.tensor(seq['hist_c'], dtype=torch.float32).unsqueeze(0)
        fut_c = torch.tensor(seq['fut_c'], dtype=torch.float32).unsqueeze(0)

        hist_b_n = b_scaler.transform(hist_b).numpy().flatten()
        hist_c_n = c_scaler.transform(hist_c.reshape(-1, CLIMATE_DIM)).numpy().flatten()
        fut_c_n = c_scaler.transform(fut_c.reshape(-1, CLIMATE_DIM)).reshape(fut_c.shape).numpy()

        preds = []
        for h in range(FORECAST_LEN):
            X = np.concatenate([hist_b_n, hist_c_n, fut_c_n[0, h, :]])
            pred_n = models[h].predict(X.reshape(1, -1))[0]
            preds.append(pred_n)
        preds_t = torch.tensor([preds], dtype=torch.float32)
        pred = b_scaler.inverse(preds_t)[0].numpy()

        results[gid] = {
            'y_pred': pred,
            'y_true': seq['target_b'],
            'anchor_balance': float(seq['hist_b'][-1]),
        }
    return results


def predict_pdd_ti(pdd_models, test_seqs):
    """Predict using PDD/temperature-index linear models."""
    results = {}
    for seq in test_seqs:
        gid = seq['glacier_id']
        if gid in pdd_models:
            model_info = pdd_models[gid]
            preds = []
            for h in range(FORECAST_LEN):
                X = seq['fut_c'][h:h+1, :]  # (1, climate_dim)
                pred = model_info['model'].predict(X)[0]
                preds.append(pred)
            results[gid] = {
                'y_pred': np.array(preds, dtype=np.float32),
                'y_true': seq['target_b'],
                'anchor_balance': float(seq['hist_b'][-1]),
            }
        else:
            results[gid] = {
                'y_pred': np.full(FORECAST_LEN, seq['hist_b'][-1], dtype=np.float32),
                'y_true': seq['target_b'],
                'anchor_balance': float(seq['hist_b'][-1]),
            }
    return results


def compute_metrics(results, region_lookup=None):
    """Pooled and region-balanced RMSE, MAE and WMAPE from a results dict.

    WMAPE (weighted MAPE, also WAPE or the MAD/Mean ratio) is
    sum|y - yhat| / sum|y|: MAE expressed as a fraction of the mean absolute
    observed balance. Plain MAPE is unusable on this target. Annual balance
    changes sign at the equilibrium line, so a near-balanced glacier-year puts
    a near-zero |y| in a per-point denominator and the ratio diverges. Across
    every 2019+ observation in the master table, 9.6% sit within 0.25 m w.e. of
    zero and 3.3% within 0.10 m w.e. Pooling the
    numerator and denominator keeps those years contributing their error
    without contributing a blow-up.

    The denominator is a property of the test set, not of the model, so on a
    fixed split WMAPE ranks models exactly as MAE does. It is reported to put
    the error on a scale-free footing, not as an independent test.
    """
    all_errors = []
    per_glacier = {}
    per_glacier_abs = {}
    for gid, r in results.items():
        err = (r['y_pred'] - r['y_true']) ** 2
        all_errors.extend(err.tolist())
        per_glacier[gid] = float(np.sqrt(np.mean(err)))
        resid = (np.asarray(r['y_pred'], dtype=np.float64).reshape(-1)
                 - np.asarray(r['y_true'], dtype=np.float64).reshape(-1))
        truth = np.asarray(r['y_true'], dtype=np.float64).reshape(-1)
        per_glacier_abs[gid] = (float(np.abs(resid).sum()), float(np.abs(truth).sum()))

    pooled_rmse = float(np.sqrt(np.mean(all_errors)))
    abs_err_sum = float(sum(v[0] for v in per_glacier_abs.values()))
    abs_true_sum = float(sum(v[1] for v in per_glacier_abs.values()))
    pooled_mae = abs_err_sum / max(len(all_errors), 1)
    pooled_wmape = abs_err_sum / abs_true_sum if abs_true_sum > 0 else float('nan')

    # Region-balanced RMSE
    region_bal = pooled_rmse  # fallback
    region_bal_wmape = pooled_wmape  # fallback
    if region_lookup:
        region_rmses = {}
        region_abs = {}
        for gid, rmse in per_glacier.items():
            reg = region_lookup.get(gid, 'unknown')
            if reg not in region_rmses:
                region_rmses[reg] = []
                region_abs[reg] = [0.0, 0.0]
            region_rmses[reg].append(rmse)
            region_abs[reg][0] += per_glacier_abs[gid][0]
            region_abs[reg][1] += per_glacier_abs[gid][1]
        region_means = [np.mean(v) for v in region_rmses.values()]
        region_bal = float(np.mean(region_means))
        # WMAPE is pooled within a region before averaging across regions: a
        # per-glacier ratio would reintroduce the small-denominator instability
        # that pooling exists to remove.
        reg_w = [e / t for e, t in region_abs.values() if t > 0]
        if reg_w:
            region_bal_wmape = float(np.mean(reg_w))

    return {
        'pooled_rmse': pooled_rmse,
        'region_balanced_rmse': region_bal,
        'pooled_mae': pooled_mae,
        'pooled_wmape': pooled_wmape,
        'region_balanced_wmape': region_bal_wmape,
        'per_glacier': per_glacier,
        'n_glaciers': len(results),
    }


def naive_persistence_results(test_seqs):
    """Naive baseline: repeat last known balance."""
    results = {}
    for seq in test_seqs:
        gid = seq['glacier_id']
        anchor = float(seq['hist_b'][-1])
        results[gid] = {
            'y_pred': np.full(FORECAST_LEN, anchor, dtype=np.float32),
            'y_true': seq['target_b'],
            'anchor_balance': anchor,
        }
    return results


print("Prediction infrastructure ready.")


Prediction infrastructure ready.


## 5. Build Populations & Report Splits

In [13]:
# ============================================================================
# Build train / holdout populations + sequences + test windows
# ============================================================================

master = df.copy()
train_pop = master[master['role'] == 'training_population'].copy()
holdout_pop = master[master['role'] == 'external_holdout'].copy()

region_lookup = (
    master.drop_duplicates('glacier_id')
    .set_index('glacier_id')['rgi_region'].to_dict()
)

train_glacier_ids = sorted(train_pop['glacier_id'].unique().tolist())
holdout_glacier_ids = sorted(holdout_pop['glacier_id'].unique().tolist())

n_train_glaciers = len(train_glacier_ids)
n_holdout_glaciers = len(holdout_glacier_ids)
train_regions = sorted(train_pop['rgi_region'].dropna().unique())
holdout_regions = sorted(holdout_pop['rgi_region'].dropna().unique())

print(f"Training: {n_train_glaciers} glaciers across {len(train_regions)} regions")
print(f"  Regions: {train_regions}")
print(f"Holdout:  {n_holdout_glaciers} glaciers across {len(holdout_regions)} regions")
print(f"  Regions: {holdout_regions}")
print()
for gid in holdout_glacier_ids:
    row = holdout_pop[holdout_pop['glacier_id'] == gid].iloc[0]
    gname = row.get('glacier_name', f'ID-{gid}')
    print(f"  Holdout: {gname} (glacier_id={gid}, RGI-{row['rgi_region']})")

# ── Build ALL training sequences (years <= 2018 only) ─────────────────────
# Filter training data to years <= 2018 for fitting
train_fit = train_pop[train_pop['year'] <= 2018].copy()
all_train_seqs = build_sequences(train_fit, train_glacier_ids, max_estimated_years=2)

# ── IID Temporal Test: last valid window per training glacier with fut_years > 2018 ──
# Use full data but only take windows where forecast starts after 2018
iid_temporal_seqs = build_test_window(train_pop, train_glacier_ids)
# Filter to only those whose forecast years are post-2018
iid_temporal_seqs = [s for s in iid_temporal_seqs if min(s['fut_years']) >= 2019]

# ── OOD Holdout Test: last valid window per holdout glacier ───────────────
ood_test_seqs = build_test_window(holdout_pop, holdout_glacier_ids)

print(f"\n{'='*60}")
print(f"Training sequences:        {len(all_train_seqs):>5d}  (years <= 2018)")
print(f"IID temporal test glaciers: {len(iid_temporal_seqs):>4d}  (independent units, fut >= 2019)")
print(f"OOD holdout test glaciers:  {len(ood_test_seqs):>4d}  (independent units)")
print(f"{'='*60}")

# Report holdout pass/fail
print("\nHoldout test-window availability:")
holdout_pass = set(s['glacier_id'] for s in ood_test_seqs)
for gid in holdout_glacier_ids:
    row = holdout_pop[holdout_pop['glacier_id'] == gid].iloc[0]
    gname = row.get('glacier_name', f'ID-{gid}')
    status = "PASS" if gid in holdout_pass else "FAIL (insufficient data)"
    print(f"  {gname:<30s} [{status}]")


Training: 101 glaciers across 12 regions
  Regions: [np.int64(1), np.int64(2), np.int64(3), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17)]
Holdout:  20 glaciers across 7 regions
  Regions: [np.int64(5), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]

  Holdout: YANAMAREY (glacier_id=226, RGI-16)
  Holdout: ROLLESTON (glacier_id=1538, RGI-18)
  Holdout: BREWSTER (glacier_id=1597, RGI-18)
  Holdout: ANTIZANA15ALPHA (glacier_id=1624, RGI-16)
  Holdout: MITTIVAKKAT (glacier_id=1629, RGI-5)
  Holdout: MARTIAL ESTE (glacier_id=2000, RGI-17)
  Holdout: BAHIA DEL DIABLO (glacier_id=2665, RGI-19)
  Holdout: CHARQUINI SUR (glacier_id=2667, RGI-16)
  Holdout: CONEJERAS (glacier_id=2721, RGI-16)
  Holdout: CHHOTA SHIGRI (glacier_id=2921, RGI-14)
  Holdout: FREYA (glacier_id=3350, RGI-5)
  Holdout: JOHNSONS (glacier_id=3366, RGI-19)
  Holdout: HURD (glacier_id=3367, RGI-19)
  Holdout: CO


Training sequences:         2259  (years <= 2018)
IID temporal test glaciers:   65  (independent units, fut >= 2019)
OOD holdout test glaciers:    15  (independent units)

Holdout test-window availability:
  YANAMAREY                      [FAIL (insufficient data)]
  ROLLESTON                      [PASS]
  BREWSTER                       [PASS]
  ANTIZANA15ALPHA                [PASS]
  MITTIVAKKAT                    [FAIL (insufficient data)]
  MARTIAL ESTE                   [PASS]
  BAHIA DEL DIABLO               [PASS]
  CHARQUINI SUR                  [PASS]
  CONEJERAS                      [PASS]
  CHHOTA SHIGRI                  [PASS]
  FREYA                          [PASS]
  JOHNSONS                       [PASS]
  HURD                           [PASS]
  CONCONTA NORTE                 [FAIL (insufficient data)]
  BROWN SUPERIOR                 [FAIL (insufficient data)]
  MOCHO CHOSHUENCO SE            [PASS]
  PARLUNG NO. 94                 [PASS]
  MERA                           

## 6. Train All Models

In [14]:
print("=" * 70)
print("TRAINING PIKA (CT-Koopman + ODO + DiffTI + Quantiles + Unrolling)")
print("=" * 70)

config = TrainConfig(
    epochs=1200, lr=1e-3,
    latent_dim=16, hidden=32,
    physics_weight=0.02, lyapunov_weight=0.01,
    use_quantiles=True, use_physics=True,
    seed=0,
)
pikav2_model, pikav2_scalers, pikav2_history = train_pikav2(
    all_train_seqs, config, verbose=True,
)
print(f"\nFinal spectral radius: {pikav2_model.spectral_info()['spectral_radius']:.4f}")
n_params_pika = sum(p.numel() for p in pikav2_model.parameters())
print(f"Parameters: {n_params_pika:,}")


TRAINING PIKA (CT-Koopman + ODO + DiffTI + Quantiles + Unrolling)


  Epoch    0 | loss=1.5240 mse=1.5735 phys=0.0000 rho=0.701


  Epoch  200 | loss=0.4754 mse=0.4512 phys=0.8896 rho=0.736


  Epoch  400 | loss=0.4099 mse=0.3798 phys=0.9983 rho=0.746


  Epoch  600 | loss=0.3780 mse=0.3464 phys=1.0183 rho=0.751


  Epoch  800 | loss=0.3617 mse=0.3290 phys=1.0366 rho=0.754


  Epoch 1000 | loss=0.3551 mse=0.3221 phys=1.0443 rho=0.755


  Epoch 1199 | loss=0.3541 mse=0.3209 phys=1.0455 rho=0.755

Final spectral radius: 0.7550
Parameters: 5,852


In [15]:
print("\n" + "=" * 70)
print("TRAINING LSTM BASELINE")
print("=" * 70)

lstm_model, lstm_scalers, lstm_history = train_lstm_baseline(
    all_train_seqs, epochs=900, hidden=64, seed=0, verbose=True,
)
n_params_lstm = sum(p.numel() for p in lstm_model.parameters())
print(f"\nParameters: {n_params_lstm:,}")



TRAINING LSTM BASELINE
  Epoch    0 | loss=1.1472


  Epoch  200 | loss=0.3608


  Epoch  400 | loss=0.2777


  Epoch  600 | loss=0.2321


  Epoch  800 | loss=0.2188


  Epoch  899 | loss=0.2183

Parameters: 51,265


In [16]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

# ── XGBoost (per-horizon GBR) ─────────────────────────────────────────────

def train_xgboost_baseline(train_seqs, seed=0):
    """Train one GradientBoostingRegressor per horizon step."""
    np.random.seed(seed)
    hist_b, hist_c, fut_c, target_b = seqs_to_tensors(train_seqs)
    b_scaler = Scaler(hist_b, scalar=True)
    c_all = torch.cat([hist_c, fut_c], dim=1)
    c_scaler = Scaler(c_all.reshape(-1, CLIMATE_DIM), scalar=False)

    hist_b_n = b_scaler.transform(hist_b).numpy()
    hist_c_n = c_scaler.transform(hist_c.reshape(-1, CLIMATE_DIM)).numpy().reshape(len(train_seqs), -1)
    fut_c_n = c_scaler.transform(fut_c.reshape(-1, CLIMATE_DIM)).reshape(fut_c.shape).numpy()
    target_n = b_scaler.transform(target_b).numpy()

    models = []
    for h in range(FORECAST_LEN):
        X = np.hstack([hist_b_n, hist_c_n, fut_c_n[:, h, :]])
        y = target_n[:, h]
        gbr = GradientBoostingRegressor(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, random_state=seed,
        )
        gbr.fit(X, y)
        models.append(gbr)
    print("  [OK] XGBoost trained (5 horizon models)")
    return models, (b_scaler, c_scaler)


# ── Random Forest (per-horizon) ──────────────────────────────────────────

def train_rf_baseline(train_seqs, seed=0):
    """Train one RandomForestRegressor per horizon step."""
    np.random.seed(seed)
    hist_b, hist_c, fut_c, target_b = seqs_to_tensors(train_seqs)
    b_scaler = Scaler(hist_b, scalar=True)
    c_all = torch.cat([hist_c, fut_c], dim=1)
    c_scaler = Scaler(c_all.reshape(-1, CLIMATE_DIM), scalar=False)

    hist_b_n = b_scaler.transform(hist_b).numpy()
    hist_c_n = c_scaler.transform(hist_c.reshape(-1, CLIMATE_DIM)).numpy().reshape(len(train_seqs), -1)
    fut_c_n = c_scaler.transform(fut_c.reshape(-1, CLIMATE_DIM)).reshape(fut_c.shape).numpy()
    target_n = b_scaler.transform(target_b).numpy()

    models = []
    for h in range(FORECAST_LEN):
        X = np.hstack([hist_b_n, hist_c_n, fut_c_n[:, h, :]])
        y = target_n[:, h]
        rf = RandomForestRegressor(
            n_estimators=200, max_depth=8,
            random_state=seed, n_jobs=-1,
        )
        rf.fit(X, y)
        models.append(rf)
    print("  [OK] Random Forest trained (5 horizon models)")
    return models, (b_scaler, c_scaler)


# ── PDD / Temperature-Index ──────────────────────────────────────────────

def train_pdd_baseline(train_seqs):
    """Per-glacier linear regression: balance ~ PDD + solid_precip + summer_temp."""
    from collections import defaultdict
    glacier_seqs = defaultdict(list)
    for s in train_seqs:
        glacier_seqs[s['glacier_id']].append(s)

    pdd_models = {}
    for gid, seqs in glacier_seqs.items():
        X_list, y_list = [], []
        for s in seqs:
            for h in range(FORECAST_LEN):
                X_list.append(s['fut_c'][h, :])
                y_list.append(s['target_b'][h])
        X = np.array(X_list, dtype=np.float32)
        y = np.array(y_list, dtype=np.float32)
        if len(X) >= 5:
            lr = LinearRegression()
            lr.fit(X, y)
            pdd_models[gid] = {'model': lr}

    print(f"  [OK] PDD/TI models fitted for {len(pdd_models)} glaciers")
    return pdd_models


# ── Larger PIKA (48/96 capacity control; not assumed worse) ─────────

def train_reduced_pikav2(train_seqs, seed=0):
    """Train a *larger* PIKA (48/96) as a capacity control (not assumed worse)."""
    cfg = TrainConfig(
        epochs=1200, lr=1e-3,
        latent_dim=48, hidden=96,
        physics_weight=0.02, lyapunov_weight=0.01,
        use_quantiles=True, use_physics=True,
        seed=seed,
    )
    model, scalers, history = train_pikav2(train_seqs, cfg, verbose=False)
    n_p = sum(p.numel() for p in model.parameters())
    print(f"  [OK] PIKA (large 48/96) trained ({n_p:,} params)")
    return model, scalers, history


# ── Train all baselines ──────────────────────────────────────────────────
print("=" * 70)
print("TRAINING BASELINES")
print("=" * 70)

xgb_models, xgb_scalers = train_xgboost_baseline(all_train_seqs, seed=0)
rf_models, rf_scalers = train_rf_baseline(all_train_seqs, seed=0)
pdd_models = train_pdd_baseline(all_train_seqs)
reduced_model, reduced_scalers, reduced_history = train_reduced_pikav2(all_train_seqs, seed=0)

print("\nAll baselines trained successfully.")


TRAINING BASELINES


  [OK] XGBoost trained (5 horizon models)


  [OK] Random Forest trained (5 horizon models)
  [OK] PDD/TI models fitted for 88 glaciers


  [OK] PIKA (large 48/96) trained (35,068 params)

All baselines trained successfully.


## 7. Evaluation Results

In [17]:
# ============================================================================
# FULL EVALUATION — IID TEMPORAL + OOD HOLDOUT
# ============================================================================

# Generate predictions
pikav2_iid = predict_pikav2(pikav2_model, pikav2_scalers, iid_temporal_seqs)
lstm_iid = predict_lstm(lstm_model, lstm_scalers, iid_temporal_seqs)
xgb_iid = predict_sklearn(xgb_models, xgb_scalers, iid_temporal_seqs)
rf_iid = predict_sklearn(rf_models, rf_scalers, iid_temporal_seqs)
pdd_iid = predict_pdd_ti(pdd_models, iid_temporal_seqs)
reduced_iid = predict_pikav2(reduced_model, reduced_scalers, iid_temporal_seqs)
naive_iid = naive_persistence_results(iid_temporal_seqs)

pikav2_ood = predict_pikav2(pikav2_model, pikav2_scalers, ood_test_seqs)
lstm_ood = predict_lstm(lstm_model, lstm_scalers, ood_test_seqs)
xgb_ood = predict_sklearn(xgb_models, xgb_scalers, ood_test_seqs)
rf_ood = predict_sklearn(rf_models, rf_scalers, ood_test_seqs)
pdd_ood = predict_pdd_ti(pdd_models, ood_test_seqs)
reduced_ood = predict_pikav2(reduced_model, reduced_scalers, ood_test_seqs)
naive_ood = naive_persistence_results(ood_test_seqs)

# Persistence blend: ŷ = w * PIKA + (1-w) * last_balance.
# Do NOT pick w by pooled IID RMSE — PIKA already beats naive there, so w*→1
# and OOD never moves. Instead: per-glacier MSE-optimal w clipped to [0,1],
# then equal-average across RGI regions (same independent-unit logic as conformal).
def _blend(model_preds, naive_preds, w):
    out = {}
    for gid, d in model_preds.items():
        rec = dict(d)
        rec['y_pred'] = w * d['y_pred'] + (1.0 - w) * naive_preds[gid]['y_pred']
        out[gid] = rec
    return out

def _region_equal_blend_w(model_preds, naive_preds, region_lookup):
    per_g = {}
    for gid, d in model_preds.items():
        p = np.asarray(d['y_pred'], dtype=np.float64).reshape(-1)
        n = np.asarray(naive_preds[gid]['y_pred'], dtype=np.float64).reshape(-1)
        y = np.asarray(d['y_true'], dtype=np.float64).reshape(-1)
        dpn = p - n
        denom = float(np.dot(dpn, dpn)) + 1e-12
        per_g[gid] = float(np.clip(np.dot(dpn, y - n) / denom, 0.0, 1.0))
    by_reg = {}
    for gid, w_g in per_g.items():
        by_reg.setdefault(region_lookup.get(gid, 'unk'), []).append(w_g)
    w_star = float(np.mean([np.mean(v) for v in by_reg.values()]))
    return w_star, per_g, {r: float(np.mean(v)) for r, v in by_reg.items()}

best_w, _per_g_w, _reg_w = _region_equal_blend_w(pikav2_iid, naive_iid, region_lookup)
print(f"IID region-equal persistence blend: w*={best_w:.2f} (1=pure PIKA, 0=naive)")
print("  per-region mean w:", {str(k): round(v, 2) for k, v in sorted(_reg_w.items(), key=lambda x: str(x[0]))})
pikav2_ood_blend = _blend(pikav2_ood, naive_ood, best_w)
print(f"  OOD pooled at w*: {compute_metrics(pikav2_ood_blend)['pooled_rmse']:.4f} "
      f"(raw PIKA {compute_metrics(pikav2_ood)['pooled_rmse']:.4f}, naive {compute_metrics(naive_ood)['pooled_rmse']:.4f})")

# Compute metrics
models_iid = {
    'PIKA': pikav2_iid,
    'LSTM': lstm_iid,
    'XGBoost': xgb_iid,
    'Random Forest': rf_iid,
    'PDD/TI': pdd_iid,
    'PIKA (large)': reduced_iid,
    'Naive Persistence': naive_iid,
}
models_ood = {
    'PIKA': pikav2_ood,
    'LSTM': lstm_ood,
    'XGBoost': xgb_ood,
    'Random Forest': rf_ood,
    'PDD/TI': pdd_ood,
    'PIKA (large)': reduced_ood,
    'PIKA + persist blend': pikav2_ood_blend,
    'Naive Persistence': naive_ood,
}

param_counts = {
    'PIKA': n_params_pika,
    'LSTM': n_params_lstm,
    'XGBoost': sum(m.n_estimators * 200 for m in xgb_models),  # approximate
    'Random Forest': sum(m.n_estimators * 200 for m in rf_models),
    'PDD/TI': len(pdd_models) * 4,
    'PIKA (large)': sum(p.numel() for p in reduced_model.parameters()),
    'Naive Persistence': 0,
}

# Print IID results
print("\n" + "=" * 80)
print("--- IID TEMPORAL TEST ---")
print(f"{'Model':<20s} {'Pooled RMSE':>12s} {'Region-Bal':>12s} {'vs Naive':>10s} {'Params':>10s}")
print("-" * 80)

naive_pooled = compute_metrics(naive_iid, region_lookup)['pooled_rmse']
for name, preds in models_iid.items():
    m = compute_metrics(preds, region_lookup)
    vs_naive = naive_pooled - m['pooled_rmse']
    params = param_counts.get(name, 0)
    print(f"{name:<20s} {m['pooled_rmse']:>12.4f} {m['region_balanced_rmse']:>12.4f} "
          f"{vs_naive:>+10.4f} {params:>10,}")

# Print OOD results
print("\n" + "=" * 80)
print("--- OOD HOLDOUT TEST ---")
print(f"{'Model':<20s} {'Pooled RMSE':>12s} {'vs Naive':>10s} {'N glaciers':>10s}")
print("-" * 80)

naive_ood_pooled = compute_metrics(naive_ood, region_lookup)['pooled_rmse']
for name, preds in models_ood.items():
    m = compute_metrics(preds, region_lookup)
    vs_naive = naive_ood_pooled - m['pooled_rmse']
    print(f"{name:<20s} {m['pooled_rmse']:>12.4f} {vs_naive:>+10.4f} {m['n_glaciers']:>10d}")

# Per-glacier OOD breakdown
print("\n  Per-glacier OOD breakdown:")
print(f"  {'Glacier':<30s} {'PIKA':>8s} {'Blend':>8s} {'LSTM':>8s} {'XGBoost':>8s} {'Naive':>8s}")
print("  " + "-" * 74)
for seq in ood_test_seqs:
    gid = seq['glacier_id']
    gname = str(holdout_pop[holdout_pop['glacier_id'] == gid]['glacier_name'].iloc[0])[:28]
    p_rmse = float(np.sqrt(np.mean((pikav2_ood[gid]['y_pred'] - pikav2_ood[gid]['y_true'])**2)))
    b_rmse = float(np.sqrt(np.mean((pikav2_ood_blend[gid]['y_pred'] - pikav2_ood_blend[gid]['y_true'])**2)))
    l_rmse = float(np.sqrt(np.mean((lstm_ood[gid]['y_pred'] - lstm_ood[gid]['y_true'])**2)))
    x_rmse = float(np.sqrt(np.mean((xgb_ood[gid]['y_pred'] - xgb_ood[gid]['y_true'])**2)))
    n_rmse = float(np.sqrt(np.mean((naive_ood[gid]['y_pred'] - naive_ood[gid]['y_true'])**2)))
    print(f"  {gname:<30s} {p_rmse:>8.3f} {b_rmse:>8.3f} {l_rmse:>8.3f} {x_rmse:>8.3f} {n_rmse:>8.3f}")


IID region-equal persistence blend: w*=0.76 (1=pure PIKA, 0=naive)
  per-region mean w: {'1': 0.5, '11': 0.94, '12': 0.81, '13': 0.98, '2': 0.56, '3': 0.76, '6': 0.94, '7': 0.38, '8': 0.97}
  OOD pooled at w*: 0.8446 (raw PIKA 0.9014, naive 0.9486)

--- IID TEMPORAL TEST ---
Model                 Pooled RMSE   Region-Bal   vs Naive     Params
--------------------------------------------------------------------------------
PIKA                    0.9332       0.6483    +0.4699      5,852
LSTM                       0.8000       0.5799    +0.6031     51,265
XGBoost                    0.8813       0.6610    +0.5218    200,000
Random Forest              1.0083       0.7531    +0.3948    200,000
PDD/TI                     0.8546       0.6346    +0.5485        352
PIKA (large)            0.8781       0.6614    +0.5250     35,068
Naive Persistence          1.4031       0.9925    +0.0000          0

--- OOD HOLDOUT TEST ---
Model                 Pooled RMSE   vs Naive N glaciers
---------------

In [18]:
# ============================================================================
# Per-Horizon RMSE (all models, horizons 1-5)
# ============================================================================

def compute_per_horizon_rmse(results):
    """Compute RMSE at each forecast horizon."""
    horizon_rmses = {}
    for h in range(FORECAST_LEN):
        errors_sq = []
        for gid, r in results.items():
            yp = np.asarray(r['y_pred']).reshape(-1)
            yt = np.asarray(r['y_true']).reshape(-1)
            if len(yp) > h and len(yt) > h:
                errors_sq.append((yp[h] - yt[h]) ** 2)
        if errors_sq:
            horizon_rmses[h + 1] = float(np.sqrt(np.mean(errors_sq)))
        else:
            horizon_rmses[h + 1] = float('nan')
    return horizon_rmses

print("\n" + "=" * 70)
print("PER-HORIZON RMSE (IID Temporal Test)")
print("=" * 70)

horizon_data = {}
for name, preds in models_iid.items():
    if preds:
        horizon_data[name] = compute_per_horizon_rmse(preds)

header = f"{'Horizon':<10}"
for name in horizon_data:
    header += f" {name[:14]:>14}"
print(header)
print("-" * len(header))

for h in range(1, FORECAST_LEN + 1):
    row = f"  h={h:<5}"
    for name, hd in horizon_data.items():
        row += f" {hd.get(h, float('nan')):>14.4f}"
    print(row)



PER-HORIZON RMSE (IID Temporal Test)
Horizon           PIKA           LSTM        XGBoost  Random Forest         PDD/TI PIKA (large Naive Persiste
-------------------------------------------------------------------------------------------------------------------
  h=1             0.6901         0.4896         0.6253         0.6746         0.4979         0.6366         1.0716
  h=2             0.9282         0.7156         0.8123         1.0601         0.7812         1.0035         1.4358
  h=3             1.0605         0.8825         1.0505         1.1937         0.8706         0.8922         1.6040
  h=4             1.0959         1.0547         1.0929         1.1719         1.1236         1.0554         1.5008
  h=5             0.8311         0.7461         0.7311         0.8405         0.8793         0.7303         1.3448


In [19]:
# ============================================================================
# WMAPE — scale-free error, sum|y - yhat| / sum|y|
# ============================================================================
# Reported alongside RMSE because the two answer different questions. RMSE is
# in m w.e. and is dominated by the handful of largest residuals; WMAPE is
# unitless and treats every metre of error the same, so a model that is steady
# everywhere and a model that is excellent except on three glaciers separate
# here in a way pooled RMSE hides.
#
# Plain MAPE is not an option on this target: annual balance changes sign at
# the equilibrium line: across every 2019+ observation in the master table,
# 9.6% lie within 0.25 m w.e. of zero, so a per-point |y| denominator diverges.
# WMAPE pools the numerator and denominator, which is exactly the fix.
#
# One caveat stated up front: the denominator depends only on the test set, so
# on a fixed split WMAPE is a constant rescaling of MAE and cannot reorder
# models relative to MAE. It reorders them relative to RMSE, which is the point.

def compute_wmape(results):
    """Pooled WMAPE over every glacier-year in a results dict."""
    num = 0.0
    den = 0.0
    for r in results.values():
        yp = np.asarray(r['y_pred'], dtype=np.float64).reshape(-1)
        yt = np.asarray(r['y_true'], dtype=np.float64).reshape(-1)
        num += float(np.abs(yp - yt).sum())
        den += float(np.abs(yt).sum())
    return num / den if den > 0 else float('nan')


def compute_per_horizon_wmape(results):
    """WMAPE at each forecast horizon, pooled across glaciers within a horizon."""
    out = {}
    for h in range(FORECAST_LEN):
        num = 0.0
        den = 0.0
        for r in results.values():
            yp = np.asarray(r['y_pred'], dtype=np.float64).reshape(-1)
            yt = np.asarray(r['y_true'], dtype=np.float64).reshape(-1)
            if len(yp) > h and len(yt) > h:
                num += abs(yp[h] - yt[h])
                den += abs(yt[h])
        out[h + 1] = num / den if den > 0 else float('nan')
    return out


print("\n" + "=" * 70)
print("WMAPE  (sum|y - yhat| / sum|y|; lower is better)")
print("=" * 70)

for label, preds_by_model in (("IID TEMPORAL", models_iid), ("OOD HOLDOUT", models_ood)):
    ref = next(iter(preds_by_model.values()))
    ys = np.concatenate([np.asarray(r['y_true'], dtype=np.float64).reshape(-1)
                         for r in ref.values()])
    print(f"\n{label} — denominator mean|y| = {np.abs(ys).mean():.3f} m w.e. "
          f"over {ys.size} glacier-years")
    print(f"  {'Model':<24} {'WMAPE':>9} {'reg-bal':>9} {'MAE':>9} {'RMSE':>9}")
    print("  " + "-" * 62)
    for name, preds in preds_by_model.items():
        if not preds:
            continue
        m = compute_metrics(preds, region_lookup)
        print(f"  {name[:24]:<24} {m['pooled_wmape']:>8.1%} "
              f"{m['region_balanced_wmape']:>8.1%} "
              f"{m['pooled_mae']:>9.4f} {m['pooled_rmse']:>9.4f}")

print("\n" + "=" * 70)
print("PER-HORIZON WMAPE (IID Temporal Test)")
print("=" * 70)
h_wmape = {name: compute_per_horizon_wmape(preds)
           for name, preds in models_iid.items() if preds}
header = f"{'Horizon':<10}"
for name in h_wmape:
    header += f" {name[:14]:>14}"
print(header)
print("-" * len(header))
for h in range(1, FORECAST_LEN + 1):
    row = f"  h={h:<5}"
    for hd in h_wmape.values():
        row += f" {hd.get(h, float('nan')):>13.1%}"
    print(row)



WMAPE  (sum|y - yhat| / sum|y|; lower is better)

IID TEMPORAL — denominator mean|y| = 1.497 m w.e. over 325 glacier-years
  Model                        WMAPE   reg-bal       MAE      RMSE
  --------------------------------------------------------------
  PIKA                     47.1%    49.9%    0.7056    0.9332
  LSTM                        40.9%    45.4%    0.6121    0.8000
  XGBoost                     45.7%    50.9%    0.6841    0.8813
  Random Forest               52.7%    57.0%    0.7887    1.0083
  PDD/TI                      40.6%    49.5%    0.6082    0.8546
  PIKA (large)             45.3%    56.8%    0.6780    0.8781
  Naive Persistence           73.9%    82.9%    1.1065    1.4031

OOD HOLDOUT — denominator mean|y| = 1.026 m w.e. over 75 glacier-years
  Model                        WMAPE   reg-bal       MAE      RMSE
  --------------------------------------------------------------
  PIKA                     69.5%    83.0%    0.7130    0.9014
  LSTM                       

## 8. Uncertainty Quantification (Raw + Conformal)

In [20]:
# ============================================================================
# Uncertainty Quantification — Raw + Conformal Calibration
# ============================================================================

print("=" * 70)
print("UNCERTAINTY QUANTIFICATION (PIKA)")
print("=" * 70)

# Collect all quantile predictions on IID test set
all_q_preds, all_targets = [], []
for gid, d in pikav2_iid.items():
    if d.get('quantiles') is not None:
        all_q_preds.append(d['quantiles'])  # (forecast_len, n_quantiles)
        all_targets.append(d['y_true'])

if all_q_preds:
    q_preds = np.vstack(all_q_preds)  # (N*forecast_len, n_quantiles) or (N, forecast_len, n_q)
    targets = np.concatenate(all_targets)

    # Flatten if needed
    if q_preds.ndim == 3:
        n_obs, n_steps, n_q = q_preds.shape
        q_preds_flat = q_preds.reshape(-1, n_q)
        targets_flat = targets.reshape(-1)
    else:
        q_preds_flat = q_preds
        targets_flat = targets

    QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]

    # --- Raw calibration ---
    print("\n--- RAW Calibration (all IID data) ---")
    print(f"{'Quantile':<12} {'Expected':>10} {'Observed':>10} {'Gap':>10}")
    print("-" * 44)
    for q_idx, tau in enumerate(QUANTILES):
        observed = float((targets_flat <= q_preds_flat[:, q_idx]).mean())
        gap = observed - tau
        print(f"  {tau:<10.2f} {tau:>10.2f} {observed:>10.3f} {gap:>+10.3f}")

    # 80% PI (q0.10 to q0.90)
    pi80_low = q_preds_flat[:, 0]
    pi80_high = q_preds_flat[:, 4]
    pi80_width_raw = float((pi80_high - pi80_low).mean())
    cov80_raw = float(((targets_flat >= pi80_low) & (targets_flat <= pi80_high)).mean())
    print(f"\n80% PI -- width: {pi80_width_raw:.3f} m w.e., coverage: {cov80_raw:.1%}")

    # 50% PI (q0.25 to q0.75)
    pi50_low = q_preds_flat[:, 1]
    pi50_high = q_preds_flat[:, 3]
    pi50_width_raw = float((pi50_high - pi50_low).mean())
    cov50_raw = float(((targets_flat >= pi50_low) & (targets_flat <= pi50_high)).mean())
    print(f"50% PI -- width: {pi50_width_raw:.3f} m w.e., coverage: {cov50_raw:.1%}")

    # CRPS approximation
    crps_raw = crps_from_quantiles(q_preds_flat, targets_flat)
    print(f"CRPS (raw): {crps_raw:.4f}")

    # ── Conformal calibration (glacier-level splitting) ──────────────────
    print("\n" + "-" * 50)
    print("CONFORMAL CALIBRATION (glacier-level split)")
    print("-" * 50)

    iid_gids = sorted(pikav2_iid.keys())
    n_cal = int(len(iid_gids) * 0.6)
    rng = np.random.default_rng(42)
    shuffled = list(iid_gids)
    rng.shuffle(shuffled)
    cal_gids = shuffled[:n_cal]
    test_gids = shuffled[n_cal:]

    print(f"Calibration glaciers: {len(cal_gids)}  (independent units)")
    print(f"Test glaciers:        {len(test_gids)}  (independent units)")

    # Nonconformity scores on calibration glaciers
    cal_scores = []
    for gid in cal_gids:
        d = pikav2_iid[gid]
        if d.get('quantiles') is not None:
            q = d['quantiles']
            if q.ndim == 2:
                low, high = q[:, 0], q[:, 4]
            else:
                low, high = q[0, :, 0], q[0, :, 4]
            for h in range(len(d['y_true'])):
                score = max(low[h] - d['y_true'][h], d['y_true'][h] - high[h])
                cal_scores.append(score)

    cal_scores = np.array(cal_scores)
    alpha = 0.20  # target 80% coverage
    q_hat = float(np.quantile(cal_scores, (1 - alpha) * (1 + 1/len(cal_scores))))
    print(f"\nConformal q_hat (alpha={alpha:.2f}): {q_hat:.4f}")

    # Evaluate on held-out test glaciers
    test_q_flat, test_y_flat = [], []
    for gid in test_gids:
        d = pikav2_iid[gid]
        if d.get('quantiles') is not None:
            q = d['quantiles']
            if q.ndim == 2:
                test_q_flat.append(q)
            else:
                test_q_flat.append(q.reshape(-1, q.shape[-1]))
            test_y_flat.append(d['y_true'].flatten())

    if test_q_flat:
        tq = np.vstack(test_q_flat)
        ty = np.concatenate(test_y_flat)

        # Conformal intervals
        conf_low = tq[:, 0] - q_hat
        conf_high = tq[:, 4] + q_hat
        conf_width = float((conf_high - conf_low).mean())
        conf_cov = float(((ty >= conf_low) & (ty <= conf_high)).mean())

        # Raw on same test glaciers
        raw_low = tq[:, 0]
        raw_high = tq[:, 4]
        raw_width_test = float((raw_high - raw_low).mean())
        raw_cov_test = float(((ty >= raw_low) & (ty <= raw_high)).mean())

        print(f"\n{'Metric':<25} {'Raw':>12} {'Conformal':>12}")
        print("-" * 51)
        print(f"{'80% Coverage':<25} {raw_cov_test:>11.1%} {conf_cov:>11.1%}")
        print(f"{'Interval width (m w.e.)':<25} {raw_width_test:>12.3f} {conf_width:>12.3f}")
        print(f"\nWidth increase for guaranteed coverage: {conf_width - raw_width_test:+.3f} m w.e.")
        if conf_cov >= 0.80:
            print("  Conformal calibration achieves target 80% coverage.")
        else:
            print(f"  WARNING: Conformal coverage {conf_cov:.1%} still below 80% target.")
else:
    print("  No quantile predictions available for UQ analysis.")


UNCERTAINTY QUANTIFICATION (PIKA)

--- RAW Calibration (all IID data) ---
Quantile       Expected   Observed        Gap
--------------------------------------------
  0.10             0.10      0.409     +0.309
  0.25             0.25      0.600     +0.350
  0.50             0.50      0.806     +0.306
  0.75             0.75      0.923     +0.173
  0.90             0.90      0.975     +0.075

80% PI -- width: 1.319 m w.e., coverage: 56.6%
50% PI -- width: 0.685 m w.e., coverage: 32.3%
CRPS (raw): 0.5338

--------------------------------------------------
CONFORMAL CALIBRATION (glacier-level split)
--------------------------------------------------
Calibration glaciers: 39  (independent units)
Test glaciers:        26  (independent units)

Conformal q_hat (alpha=0.20): 0.5828

Metric                             Raw    Conformal
---------------------------------------------------
80% Coverage                    59.2%       83.8%
Interval width (m w.e.)          1.281        2.447

Width 

## 9. Leave-One-Region-Out (LORO)

Transfer test: train on all RGI regions except one, evaluate on that region’s glaciers. Holdout-only regions (5, 14, 15, 18, 19) are excluded so LORO does not leak the OOD set. RGI-10 and RGI-17 were skipped this run (insufficient eval windows), leaving 10 folds.

This is **not** a claimed win. On this protocol LSTM is stronger in most folds; PIKA wins 1/10 (RGI-13). Report win rate and a fold-level CI, not a pooled “PIKA transfers” sentence.


In [21]:
# ============================================================================
# LORO infrastructure
# ============================================================================

# Regions represented ONLY in holdout — exclude from LORO
holdout_only_regions = set()
holdout_regions_set = set(holdout_pop['rgi_region'].dropna().unique())
train_regions_set = set(train_pop['rgi_region'].dropna().unique())
holdout_only_regions = holdout_regions_set - train_regions_set
print(f"Holdout-only regions (excluded from LORO): {sorted(holdout_only_regions) or 'none'}")

loro_eligible_regions = sorted(train_regions_set - holdout_only_regions)
print(f"LORO-eligible regions: {loro_eligible_regions} ({len(loro_eligible_regions)} total)")


def run_loro_fold(full_df, held_out_region, config):
    """Train PIKA + LSTM on all regions except held_out_region,
    test on held_out_region glaciers.
    Returns dict with metrics or None if insufficient data."""
    train_df = full_df[
        (full_df['role'] == 'training_population') &
        (full_df['rgi_region'] != held_out_region)
    ].copy()
    test_df = full_df[
        (full_df['role'] == 'training_population') &
        (full_df['rgi_region'] == held_out_region)
    ].copy()

    train_gids = sorted(train_df['glacier_id'].unique().tolist())
    test_gids = sorted(test_df['glacier_id'].unique().tolist())

    if len(train_gids) < 5 or len(test_gids) < 1:
        return None

    # Build training sequences (all years <= 2018)
    train_fit_df = train_df[train_df['year'] <= 2018]
    seqs = build_sequences(train_fit_df, train_gids, max_estimated_years=2)
    if len(seqs) < 20:
        return None

    # Build test windows for held-out region
    test_seqs = build_test_window(test_df, test_gids)
    test_seqs = [s for s in test_seqs if min(s['fut_years']) >= 2014]
    if len(test_seqs) < 1:
        return None

    # Train PIKA
    p_model, p_scl, _ = train_pikav2(seqs, config, verbose=False)
    p_preds = predict_pikav2(p_model, p_scl, test_seqs)
    p_m = compute_metrics(p_preds)

    # Train LSTM
    l_model, l_scl, _ = train_lstm_baseline(seqs, epochs=600, hidden=64, seed=config.seed, verbose=False)
    l_preds = predict_lstm(l_model, l_scl, test_seqs)
    l_m = compute_metrics(l_preds)

    # Naive
    n_preds = naive_persistence_results(test_seqs)
    n_m = compute_metrics(n_preds)

    pika_wins = 1 if p_m['pooled_rmse'] < l_m['pooled_rmse'] else 0

    return {
        'region': held_out_region,
        'n_train_glaciers': len(train_gids),
        'n_test_glaciers': len(test_seqs),
        'pika_rmse': p_m['pooled_rmse'],
        'lstm_rmse': l_m['pooled_rmse'],
        'naive_rmse': n_m['pooled_rmse'],
        'pika_wins': pika_wins,
    }


Holdout-only regions (excluded from LORO): [np.int64(5), np.int64(14), np.int64(15), np.int64(18), np.int64(19)]
LORO-eligible regions: [np.int64(1), np.int64(2), np.int64(3), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17)] (12 total)


In [22]:
# ============================================================================
# Run LORO
# ============================================================================

print("=" * 70)
print("LEAVE-ONE-REGION-OUT EVALUATION")
print("=" * 70)

loro_config = TrainConfig(
    epochs=800, lr=1e-3, latent_dim=16, hidden=32,
    physics_weight=0.02, lyapunov_weight=0.01,
    use_quantiles=True, use_physics=True, seed=0,
)

loro_results = []
for region in loro_eligible_regions:
    n_glaciers = train_pop[train_pop['rgi_region'] == region]['glacier_id'].nunique()
    print(f"\n  Holding out RGI-{region} ({n_glaciers} glaciers)...", end=" ", flush=True)
    fold = run_loro_fold(master, region, loro_config)
    if fold:
        loro_results.append(fold)
        print(f"PIKAv2={fold['pika_rmse']:.4f}  "
              f"LSTM={fold['lstm_rmse']:.4f}  Naive={fold['naive_rmse']:.4f}  "
              f"({fold['n_test_glaciers']} glaciers)")
    else:
        print("Skipped (insufficient data)")

if loro_results:
    loro_df = pd.DataFrame(loro_results)

    print("\n" + "-" * 70)
    print(f"{'Region':<10} {'PIKA':>10} {'LSTM':>10} {'Naive':>10} {'Winner':>10}")
    print("-" * 52)
    for _, row in loro_df.iterrows():
        winner = "PIKA" if row['pika_wins'] else "LSTM"
        print(f"RGI-{row['region']:<6} {row['pika_rmse']:>10.4f} {row['lstm_rmse']:>10.4f} "
              f"{row['naive_rmse']:>10.4f} {winner:>10}")

    pika_mean = loro_df['pika_rmse'].mean()
    lstm_mean = loro_df['lstm_rmse'].mean()
    total_wins = int(loro_df['pika_wins'].sum())
    total_folds = len(loro_df)

    print(f"\nOverall LORO mean RMSE:  PIKA={pika_mean:.4f}  LSTM={lstm_mean:.4f}")
    print(f"Fold-level win rate: PIKA wins {total_wins}/{total_folds} = {total_wins/total_folds:.1%}")

    # Bootstrap CI
    np.random.seed(42)
    boot_diffs = []
    for _ in range(5000):
        idx = np.random.choice(len(loro_df), size=len(loro_df), replace=True)
        d = loro_df.iloc[idx]['lstm_rmse'].mean() - loro_df.iloc[idx]['pika_rmse'].mean()
        boot_diffs.append(d)
    boot_diffs = np.array(boot_diffs)
    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
    print(f"Bootstrap 95% CI for (LSTM - PIKA) RMSE: [{ci_lo:+.4f}, {ci_hi:+.4f}]")
    sig = "Significant" if (ci_lo > 0 or ci_hi < 0) else "Not significant"
    print(f"  {sig} (CI {'excludes' if ci_lo > 0 or ci_hi < 0 else 'includes'} 0)")

    # Sign test
    from scipy.stats import binomtest
    result = binomtest(total_wins, total_folds, 0.5, alternative='greater')
    print(f"Sign test p-value: {result.pvalue:.4f}")
    print("  LORO is a transfer test, not a claimed win. LSTM is stronger on this protocol.")
else:
    print("\nNo LORO folds completed successfully.")
    loro_df = pd.DataFrame()


LEAVE-ONE-REGION-OUT EVALUATION

  Holding out RGI-1 (4 glaciers)... 

PIKAv2=0.9634  LSTM=0.5388  Naive=0.7917  (4 glaciers)

  Holding out RGI-2 (19 glaciers)... 

PIKAv2=1.4557  LSTM=0.8323  Naive=1.4801  (12 glaciers)

  Holding out RGI-3 (4 glaciers)... 

PIKAv2=0.5080  LSTM=0.3414  Naive=0.4756  (3 glaciers)

  Holding out RGI-6 (9 glaciers)... 

PIKAv2=0.3575  LSTM=0.2743  Naive=0.8945  (3 glaciers)

  Holding out RGI-7 (12 glaciers)... 

...

PIKAv2=0.3851  LSTM=0.3783  Naive=0.8136  (2 glaciers)

  Holding out RGI-13 (9 glaciers)... 

PIKAv2=0.5270  LSTM=0.5640  Naive=1.2077  (9 glaciers)

  Holding out RGI-16 (1 glaciers)... 

PIKAv2=1.0437  LSTM=0.7477  Naive=1.0815  (1 glaciers)

  Holding out RGI-17 (1 glaciers)... 

Skipped (insufficient data)

----------------------------------------------------------------------
Region        PIKA       LSTM      Naive     Winner
----------------------------------------------------
RGI-1.0        0.9634     0.5388     0.7917       LSTM
RGI-2.0        1.4557     0.8323     1.4801       LSTM
RGI-3.0        0.5080     0.3414     0.4756       LSTM
RGI-6.0        0.3575     0.2743     0.8945       LSTM
RGI-7.0        0.7545     0.6741     0.5370       LSTM
RGI-8.0        1.0419     0.8562     1.9362       LSTM
RGI-11.0       0.9152     0.8758     1.3502       LSTM
RGI-12.0       0.3851     0.3783     0.8136       LSTM
RGI-13.0       0.5270     0.5640     1.2077       PIKA
RGI-16.0       1.0437     0.7477     1.0815       LSTM

Overall LORO mean RMSE:  PIKA=0.7952  LSTM=0.6083
Fold-level win rate: PIKA wins 1/10 = 10.0%


Bootstrap 95% CI for (LSTM - PIKA) RMSE: [-0.3177, -0.0784]
  Significant (CI excludes 0)
Sign test p-value: 0.9990
  LORO is a transfer test, not a claimed win. LSTM is stronger on this protocol.


## 10. Ablation Study

Each row retrains with one piece removed. Residual learning is the only component whose removal substantially raises RMSE. Physics and Lyapunov do not buy IID RMSE; on this expanded split the 48/96 control and the no-quantile arm are *better* than full 16/32 (seed 0). Quote 16/32 as the compact model.


In [23]:
# ============================================================================
# Ablation study - 5 seeds, paired within seed
# ============================================================================
# Single-seed ablation cannot be read: PIKA's seed-to-seed sd on IID pooled RMSE
# is ~0.046, so any single-seed delta smaller than that is unreadable. Every
# variant is therefore retrained on all 5 seeds and scored as a PAIRED
# difference against the full model at the same seed. Pairing cancels the shared
# seed effect, so the paired sd is much smaller than the marginal sd and small
# real effects can still resolve.
#
# The residual ablation RETRAINS with TrainConfig(use_residual=False). The
# earlier version trained on deltas and then read the delta head as an absolute
# balance, which mostly measured the resulting ~1.2 m w.e. offset rather than
# the value of residual anchoring.

ABLATION_SEEDS = [0, 1, 2, 3, 4]

def _abl_cfg(seed, **kw):
    base = dict(epochs=1200, lr=1e-3, latent_dim=16, hidden=32,
                physics_weight=0.02, lyapunov_weight=0.01,
                use_quantiles=True, use_physics=True, use_residual=True, seed=seed)
    base.update(kw)
    return TrainConfig(**base)

ABLATION_VARIANTS = {
    "Full PIKA":         dict(),
    "No physics prior":     dict(physics_weight=0.0, use_physics=False),
    "No Lyapunov reg.":     dict(lyapunov_weight=0.0),
    "No quantile loss":     dict(use_quantiles=False),
    "PIKA (large 48/96)": dict(latent_dim=48, hidden=96),
    "No residual learning": dict(use_residual=False),
}

print("=" * 78)
print(f"ABLATION STUDY - {len(ABLATION_VARIANTS)} variants x {len(ABLATION_SEEDS)} seeds "
      f"= {len(ABLATION_VARIANTS) * len(ABLATION_SEEDS)} trainings")
print("=" * 78)

ablation_rows = []
for seed in ABLATION_SEEDS:
    print(f"\n  seed {seed}")
    for name, kw in ABLATION_VARIANTS.items():
        cfg = _abl_cfg(seed, **kw)
        print(f"    {name:<24}", end=" ", flush=True)
        model, scl, hist = train_pikav2(all_train_seqs, cfg, verbose=False)
        preds = predict_pikav2(model, scl, iid_temporal_seqs,
                               use_residual=cfg.use_residual)
        m = compute_metrics(preds, region_lookup)
        # Convergence guard: a variant that is still descending at the last epoch
        # would understate its own performance and inflate its ablation delta.
        tail = [h['mse'] for h in hist[-100:]]
        slope = (tail[-1] - tail[0]) / max(len(tail), 1)
        ablation_rows.append({
            'seed': seed, 'variant': name,
            'pooled_rmse': m['pooled_rmse'],
            'region_balanced_rmse': m['region_balanced_rmse'],
            'pooled_wmape': m['pooled_wmape'],
            'pooled_mae': m['pooled_mae'],
            'final_mse': tail[-1], 'tail_slope_per_epoch': slope,
        })
        print(f"RMSE={m['pooled_rmse']:.4f}  WMAPE={m['pooled_wmape']:.1%}"
              f"{'  [NOT CONVERGED]' if slope < -1e-5 else ''}")

ablation_df = pd.DataFrame(ablation_rows)

# --- paired analysis ---------------------------------------------------------
piv = ablation_df.pivot(index='seed', columns='variant', values='pooled_rmse')
full = piv["Full PIKA"]
seed_sd = float(full.std(ddof=1))

def paired_ci(d, n_boot=20000, seed=0):
    """Percentile bootstrap over seeds. n=5, so this is indicative, not tight."""
    d = np.asarray(d, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(d, size=(n_boot, d.size), replace=True).mean(axis=1)
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

print("\n" + "=" * 78)
print("PAIRED ABLATION (variant - full, same seed; positive = removing it HURTS)")
print("=" * 78)
print(f"  Full PIKA: {full.mean():.4f} +/- {seed_sd:.4f} over {len(full)} seeds")
print(f"\n  {'Variant removed':<24} {'mean d':>9} {'sd(d)':>8} {'95% CI':>20} {'verdict':>18}")
print("  " + "-" * 82)

abl_summary = []
for name in ABLATION_VARIANTS:
    if name == "Full PIKA":
        continue
    d = (piv[name] - full).to_numpy()
    lo, hi = paired_ci(d)
    resolved = lo > 0 or hi < 0
    verdict = ("HURTS to remove" if lo > 0 else
               "HELPS to remove" if hi < 0 else "within noise")
    abl_summary.append({'variant': name, 'mean_delta': float(d.mean()),
                        'sd_delta': float(d.std(ddof=1)), 'ci_lo': lo, 'ci_hi': hi,
                        'resolved': resolved,
                        'wins_vs_full': int((d < 0).sum()), 'n_seeds': len(d)})
    print(f"  {name:<24} {d.mean():>+9.4f} {d.std(ddof=1):>8.4f} "
          f"  [{lo:>+7.4f},{hi:>+7.4f}] {verdict:>18}")

abl_summary_df = pd.DataFrame(abl_summary)
print(f"\n  Marginal seed sd of the full model: {seed_sd:.4f}")
print(f"  Paired sd is smaller because pairing cancels the shared seed effect,")
print(f"  so a delta below the marginal sd can still resolve.")

# --- convergence audit -------------------------------------------------------
nc = ablation_df[ablation_df.tail_slope_per_epoch < -1e-5]
print("\n" + "-" * 78)
if len(nc):
    print(f"  WARNING: {len(nc)} run(s) still descending at the final epoch:")
    for _, r in nc.iterrows():
        print(f"    seed {r.seed} {r.variant}: slope {r.tail_slope_per_epoch:.2e}/epoch")
    print("  Their ablation deltas are upper bounds, not estimates.")
else:
    print("  Convergence: all runs flat over the final 100 epochs.")

ablation_df.to_csv('ablation_multiseed_rows.csv', index=False)
abl_summary_df.to_csv('ablation_multiseed_summary.csv', index=False)
print("\n  wrote ablation_multiseed_rows.csv, ablation_multiseed_summary.csv")

# Back-compat: downstream cells (fig4) expect `ablation_results` keyed by name.
ablation_results = {
    name: {'pooled_rmse': float(piv[name].mean()),
           'region_balanced_rmse': float(
               ablation_df[ablation_df.variant == name].region_balanced_rmse.mean())}
    for name in ABLATION_VARIANTS
}


ABLATION STUDY - 6 variants x 5 seeds = 30 trainings

  seed 0
    Full PIKA             

RMSE=0.9332  WMAPE=47.1%
    No physics prior         

RMSE=0.9041  WMAPE=45.6%
    No Lyapunov reg.         

RMSE=0.9332  WMAPE=47.1%
    No quantile loss         

RMSE=0.8686  WMAPE=43.4%
    PIKA (large 48/96)    

...

RMSE=0.8405  WMAPE=42.1%
    No Lyapunov reg.         

RMSE=0.8294  WMAPE=41.8%
    No quantile loss         

RMSE=0.8793  WMAPE=43.7%
    PIKA (large 48/96)    

RMSE=0.8263  WMAPE=42.5%
    No residual learning     

RMSE=0.9063  WMAPE=45.7%

PAIRED ABLATION (variant - full, same seed; positive = removing it HURTS)
  Full PIKA: 0.8446 +/- 0.0512 over 5 seeds

  Variant removed             mean d    sd(d)               95% CI            verdict
  ----------------------------------------------------------------------------------
  No physics prior           -0.0054   0.0148   [-0.0180,+0.0050]       within noise
  No Lyapunov reg.           +0.0000   0.0000   [+0.0000,+0.0000]       within noise
  No quantile loss           -0.0007   0.0427   [-0.0354,+0.0292]       within noise
  PIKA (large 48/96)      -0.0075   0.0355   [-0.0354,+0.0196]       within noise
  No residual learning       +0.0370   0.0582   [-0.0156,+0.0736]       within noise

  Marginal seed sd of the full model: 0.0512
  Paired sd is smaller because pairing cancels the shared seed effect,
  so a delta below the marginal sd can still resolve.

------------------------------------------------------------------------------
  Convergen

## 11. Complexity Accounting

PIKA (16/32) has ~6k parameters vs ~51k for the 2-layer LSTM. The comparison is **IID statistical tie + fewer parameters**, not RMSE-per-parameter or OOD transfer.


In [24]:
# ============================================================================
# Complexity accounting
# ============================================================================

print("=" * 70)
print("COMPLEXITY ACCOUNTING")
print("=" * 70)

complexities = {
    "PIKA": {
        "params": sum(p.numel() for p in pikav2_model.parameters()),
        "rmse": compute_metrics(pikav2_iid, region_lookup)['pooled_rmse'],
    },
    "LSTM": {
        "params": sum(p.numel() for p in lstm_model.parameters()),
        "rmse": compute_metrics(lstm_iid, region_lookup)['pooled_rmse'],
    },
    "PIKA (large)": {
        "params": sum(p.numel() for p in reduced_model.parameters()),
        "rmse": compute_metrics(reduced_iid, region_lookup)['pooled_rmse'],
    },
    "XGBoost": {
        "params": sum(m.n_estimators * (2 ** min(m.max_depth, 10)) for m in xgb_models),
        "rmse": compute_metrics(xgb_iid, region_lookup)['pooled_rmse'],
    },
    "Random Forest": {
        "params": sum(m.n_estimators * 50 for m in rf_models),  # approximate leaf nodes
        "rmse": compute_metrics(rf_iid, region_lookup)['pooled_rmse'],
    },
    "PDD/TI": {
        "params": len(pdd_models) * (CLIMATE_DIM + 1),
        "rmse": compute_metrics(pdd_iid, region_lookup)['pooled_rmse'],
    },
    "Naive": {
        "params": 0,
        "rmse": compute_metrics(naive_iid, region_lookup)['pooled_rmse'],
    },
}

print(f"\n{'Model':<20} {'Params':>10} {'IID RMSE':>10} {'RMSE/1k params':>16}")
print("-" * 58)
for name, c in complexities.items():
    p = c['params']
    rmse = c['rmse']
    if p > 0:
        eff = rmse / (p / 1000)
        print(f"{name:<20} {p:>10,} {rmse:>10.4f} {eff:>16.6f}")
    else:
        print(f"{name:<20} {p:>10,} {rmse:>10.4f} {'N/A':>16}")

print("\n--- What the parameter count buys ---")
print("PIKA 16/32 (~6k) is statistically tied with LSTM (~51k) on IID pooled RMSE")
print("(5-seed mean 0.844 vs 0.827; 95% CI for LSTM-PIKA includes 0; LSTM mean lower).")
print("Additional value that is actually supported:")
print("  1. Calibrated UQ (quantile head + glacier-level conformal)")
print("  2. Interpretable Koopman spectrum (real ODO eigenvalues)")
print("  3. Residual/delta learning (only ablation that raises RMSE when removed)")
print("  4. Compact model (~9x smaller than the LSTM)")
print("Not supported by this protocol:")
print("  - Transfer to unseen regimes (LORO 1/10 vs LSTM; CI against PIKA)")
print("  - 16/32 as pooled OOD winner (48/96 wins; 16/32 only edges LSTM/naive)")
print("  - Strongest 1-year horizon (LSTM wins every horizon on seed-0 IID)")
print("  - Physics or Lyapunov as RMSE improvements (ablation ~0 or negative)")


COMPLEXITY ACCOUNTING

Model                    Params   IID RMSE   RMSE/1k params
----------------------------------------------------------
PIKA                   5,852     0.9332         0.159463
LSTM                     51,265     0.8000         0.015604
PIKA (large)          35,068     0.8781         0.025041
XGBoost                  16,000     0.8813         0.055079
Random Forest            50,000     1.0083         0.020167
PDD/TI                      352     0.8546         2.427830
Naive                         0     1.4031              N/A

--- What the parameter count buys ---
PIKA 16/32 (~6k) is statistically tied with LSTM (~51k) on IID pooled RMSE
(5-seed mean 0.844 vs 0.827; 95% CI for LSTM-PIKA includes 0; LSTM mean lower).
Additional value that is actually supported:
  1. Calibrated UQ (quantile head + glacier-level conformal)
  2. Interpretable Koopman spectrum (real ODO eigenvalues)
  3. Residual/delta learning (only ablation that raises RMSE when removed)
  4. Compa

## 12. Multi-Seed Robustness

Five independent seeds on the frozen 16/32 recipe vs the same LSTM. Report mean ± sd and a bootstrap CI on the mean difference. **Do not quote seed 0 as the paper result.**


In [25]:
# ============================================================================
# Multi-seed robustness check (5 seeds)
# ============================================================================

SEEDS = [0, 1, 2, 3, 4]

print("=" * 70)
print("MULTI-SEED ROBUSTNESS (5 seeds)")
print("=" * 70)

seed_results = {"PIKA": [], "LSTM": []}

for seed in SEEDS:
    print(f"\n  Seed {seed}...", end=" ", flush=True)

    cfg = TrainConfig(
        epochs=1200, lr=1e-3, latent_dim=16, hidden=32,
        physics_weight=0.02, lyapunov_weight=0.01,
        use_quantiles=True, use_physics=True, seed=seed,
    )
    p_model, p_scl, _ = train_pikav2(all_train_seqs, cfg, verbose=False)
    p_preds = predict_pikav2(p_model, p_scl, iid_temporal_seqs)
    p_m = compute_metrics(p_preds, region_lookup)
    seed_results["PIKA"].append(p_m['pooled_rmse'])

    l_model, l_scl, _ = train_lstm_baseline(all_train_seqs, epochs=900, hidden=64, seed=seed, verbose=False)
    l_preds = predict_lstm(l_model, l_scl, iid_temporal_seqs)
    l_m = compute_metrics(l_preds, region_lookup)
    seed_results["LSTM"].append(l_m['pooled_rmse'])

    print(f"PIKAv2={p_m['pooled_rmse']:.4f}  LSTM={l_m['pooled_rmse']:.4f}")

# Report
print("\n" + "-" * 50)
for name in ["PIKA", "LSTM"]:
    vals = np.array(seed_results[name])
    print(f"{name:<12}  mean={vals.mean():.4f} +/- {vals.std():.4f}  "
          f"[{vals.min():.4f}, {vals.max():.4f}]")

pika_arr = np.array(seed_results["PIKA"])
lstm_arr = np.array(seed_results["LSTM"])
wins = int((pika_arr < lstm_arr).sum())
print(f"\nPIKA wins {wins}/{len(SEEDS)} seeds")

# Bootstrap CI on mean difference
np.random.seed(42)
boot_diffs = []
for _ in range(5000):
    idx = np.random.choice(len(SEEDS), size=len(SEEDS), replace=True)
    boot_diffs.append(lstm_arr[idx].mean() - pika_arr[idx].mean())
boot_diffs = np.array(boot_diffs)
ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
print(f"95% CI for (LSTM - PIKA) mean RMSE: [{ci_lo:+.4f}, {ci_hi:+.4f}]")
if ci_lo < 0 < ci_hi:
    print("  CI includes 0: IID pooled RMSE is statistically tied.")
else:
    print("  CI excludes 0.")
print("  LSTM has lower seed-to-seed variance; PIKA uses ~9x fewer parameters.")
print("  Paper claim: compact residual Koopman, not a significant IID RMSE win.")


MULTI-SEED ROBUSTNESS (5 seeds)

  Seed 0... 

PIKAv2=0.9332  LSTM=0.8000

  Seed 1... 

PIKAv2=0.8268  LSTM=0.7904

  Seed 2... 

PIKAv2=0.8337  LSTM=0.8244

  Seed 3... 

PIKAv2=0.8000  LSTM=0.8846

  Seed 4... 

PIKAv2=0.8294  LSTM=0.8122

--------------------------------------------------
PIKA       mean=0.8446 +/- 0.0458  [0.8000, 0.9332]
LSTM          mean=0.8223 +/- 0.0332  [0.7904, 0.8846]

PIKA wins 1/5 seeds
95% CI for (LSTM - PIKA) mean RMSE: [-0.0852, +0.0411]
  CI includes 0: IID pooled RMSE is statistically tied.
  LSTM has lower seed-to-seed variance; PIKA uses ~9x fewer parameters.
  Paper claim: compact residual Koopman, not a significant IID RMSE win.


## 13. Visualization

In [26]:
# Figure 1 — Training curves (2x2: loss, MSE, physics, spectral radius)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

history_df = pd.DataFrame(pikav2_history)

axes[0, 0].plot(history_df['epoch'], history_df['loss'], 'b-', alpha=0.7)
axes[0, 0].set_title('Total Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history_df['epoch'], history_df['mse'], 'r-', alpha=0.7)
axes[0, 1].set_title('MSE (normalized delta)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_yscale('log')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history_df['epoch'], history_df['physics'], 'g-', alpha=0.7)
axes[1, 0].set_title('Physics Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history_df['epoch'], history_df['spectral_radius'], 'm-', alpha=0.7)
axes[1, 1].set_title('Spectral Radius')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].axhline(y=0.95, color='k', linestyle='--', alpha=0.5, label='rho_max')
axes[1, 1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='collapse threshold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Figure 1: PIKA Training Dynamics', fontsize=14)
plt.tight_layout()
plt.savefig('fig1_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig1_training_curves.png")


Saved: fig1_training_curves.png


In [27]:
# Figure 2 — Model comparison (Region-Balanced RMSE, IID + OOD)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

model_names = list(models_iid.keys())
iid_rmses = [compute_metrics(models_iid[n], region_lookup)['region_balanced_rmse'] for n in model_names]
ood_rmses = [compute_metrics(models_ood[n], region_lookup)['pooled_rmse'] for n in model_names]

colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800', '#607D8B', '#795548']

x = np.arange(len(model_names))
bars1 = axes[0].bar(x, iid_rmses, color=colors[:len(model_names)], alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[0].set_ylabel('Region-Balanced RMSE (m w.e.)')
axes[0].set_title('IID Temporal Test')
axes[0].grid(True, alpha=0.3, axis='y')

bars2 = axes[1].bar(x, ood_rmses, color=colors[:len(model_names)], alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[1].set_ylabel('Pooled RMSE (m w.e.)')
axes[1].set_title('OOD Holdout Test')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Figure 2: Model Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('fig2_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig2_model_comparison.png")


Saved: fig2_model_comparison.png


In [28]:
# Figure 3 — Per-Horizon RMSE
fig, ax = plt.subplots(figsize=(8, 5))

selected = ['PIKA', 'LSTM', 'XGBoost', 'Naive Persistence']
colors_sel = ['#2196F3', '#FF5722', '#4CAF50', '#795548']
horizons = list(range(1, FORECAST_LEN + 1))

for name, color in zip(selected, colors_sel):
    if name in models_iid:
        h_rmse = compute_per_horizon_rmse(models_iid[name])
        ax.plot(horizons, [h_rmse[h] for h in horizons], 'o-', color=color, label=name, linewidth=2)

ax.set_xlabel('Forecast Horizon (years)')
ax.set_ylabel('RMSE (m w.e.)')
ax.set_title('Figure 3: Per-Horizon Forecast Accuracy (IID)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(horizons)
plt.tight_layout()
plt.savefig('fig3_per_horizon.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig3_per_horizon.png")


Saved: fig3_per_horizon.png


In [29]:
# Figure 4 — Ablation bar chart
fig, ax = plt.subplots(figsize=(10, 5))

abl_names = list(ablation_results.keys())
abl_rmses = [ablation_results[n]['pooled_rmse'] for n in abl_names]
# 5-seed means now, so show the spread: a bar chart of means alone would imply
# the differences are readable when most of them are not.
abl_sds = [float(ablation_df[ablation_df.variant == n].pooled_rmse.std(ddof=1))
           for n in abl_names]

x = np.arange(len(abl_names))
bars = ax.bar(x, abl_rmses, yerr=abl_sds, capsize=4,
              color=['#2196F3'] + ['#90CAF9'] * (len(abl_names) - 1), alpha=0.8,
              error_kw=dict(ecolor='#37474F', lw=1.2))
bars[0].set_color('#2196F3')  # Full model highlighted

# Add LSTM reference line
ax.axhline(y=compute_metrics(lstm_iid, region_lookup)['pooled_rmse'],
           color='#FF5722', linestyle='--', linewidth=2, label='LSTM baseline')

ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in abl_names], fontsize=8)
ax.set_ylabel('Pooled RMSE (m w.e.)')
ax.set_title('Figure 4: Ablation Study (5 seeds, error bars = 1 s.d.)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('fig4_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig4_ablation.png")


Saved: fig4_ablation.png


In [30]:
# Figure 5 — Uncertainty fan chart (1 IID + 1 OOD glacier)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def plot_fan(ax, seq_data, title, q_hat_val=None):
    """Plot fan chart with raw + conformal intervals."""
    y_true = seq_data['y_true']
    y_pred = seq_data['y_pred']
    quantiles = seq_data.get('quantiles')
    horizons = np.arange(1, len(y_true) + 1)

    ax.plot(horizons, y_true, 'ko-', label='Observed', linewidth=2, markersize=6)
    ax.plot(horizons, y_pred, 's-', color='#2196F3', label='PIKA median', linewidth=2)

    if quantiles is not None:
        q = quantiles if quantiles.ndim == 2 else quantiles[0]
        # Raw 80% PI
        ax.fill_between(horizons, q[:, 0], q[:, 4],
                       alpha=0.2, color='#2196F3', label='Raw 80% PI')
        # Raw 50% PI
        ax.fill_between(horizons, q[:, 1], q[:, 3],
                       alpha=0.3, color='#2196F3', label='Raw 50% PI')
        # Conformal 80% PI
        if q_hat_val is not None:
            ax.fill_between(horizons, q[:, 0] - q_hat_val, q[:, 4] + q_hat_val,
                           alpha=0.1, color='red', label=f'Conformal 80% PI')

    ax.set_xlabel('Forecast Horizon (years)')
    ax.set_ylabel('Mass Balance (m w.e.)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(horizons)

# Pick one IID and one OOD glacier
iid_example = list(pikav2_iid.values())[0] if pikav2_iid else None
ood_example = list(pikav2_ood.values())[0] if pikav2_ood else None

# Compute q_hat (reuse from UQ cell if available)
try:
    q_hat_val
except NameError:
    q_hat_val = 0.3  # fallback

if iid_example:
    iid_gid = list(pikav2_iid.keys())[0]
    plot_fan(axes[0], iid_example, f'IID Glacier (id={iid_gid})', q_hat_val)
if ood_example:
    ood_gid = list(pikav2_ood.keys())[0]
    plot_fan(axes[1], ood_example, f'OOD Glacier (id={ood_gid})', q_hat_val)

plt.suptitle('Figure 5: Uncertainty Quantification', fontsize=14)
plt.tight_layout()
plt.savefig('fig5_uncertainty_fan.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig5_uncertainty_fan.png")


Saved: fig5_uncertainty_fan.png


In [31]:
# Figure 6 — LORO per-region comparison
if len(loro_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))

    regions = loro_df['region'].tolist()
    x = np.arange(len(regions))
    width = 0.25

    ax.bar(x - width, loro_df['pika_rmse'].values, width,
           label='PIKA', color='#2196F3', alpha=0.8)
    ax.bar(x, loro_df['lstm_rmse'].values, width,
           label='LSTM', color='#FF5722', alpha=0.8)
    ax.bar(x + width, loro_df['naive_rmse'].values, width,
           label='Naive', color='#795548', alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels([f'RGI-{r}' for r in regions])
    ax.set_ylabel('RMSE (m w.e.)')
    ax.set_title('Figure 6: Leave-One-Region-Out Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('fig6_loro.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: fig6_loro.png")
else:
    print("No LORO results to plot.")


Saved: fig6_loro.png


In [32]:
# Figure 7 — Spectral analysis (eigenvalue magnitudes of K)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

with torch.no_grad():
    info = pikav2_model.spectral_info()
    eigs = info['eigenvalues']

# Bar chart of |exp(lambda)|
axes[0].bar(range(len(eigs)), sorted(eigs, reverse=True), color='#2196F3', alpha=0.7)
axes[0].axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='|z|=1 (stability)')
axes[0].axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='collapse threshold')
axes[0].set_xlabel('Eigenvalue Index')
axes[0].set_ylabel('|exp(lambda * dt)|')
axes[0].set_title('Koopman Operator Eigenvalue Magnitudes')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram of eigenvalue magnitudes
axes[1].hist(eigs, bins=20, color='#4CAF50', alpha=0.7, edgecolor='black')
axes[1].axvline(x=1.0, color='r', linestyle='--', alpha=0.5, label='|z|=1')
axes[1].set_xlabel('|exp(lambda * dt)|')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Eigenvalue Magnitudes')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Figure 7: Spectral Analysis of Learned Koopman Operator', fontsize=13)
plt.tight_layout()
plt.savefig('fig7_spectral.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig7_spectral.png")


Saved: fig7_spectral.png


## 14. Summary & Key Findings

**Protocol.** Training years ≤ 2018; IID test = last strict 2019+ window per training glacier (**65** units). Assigned holdouts = **20**; OOD eval = **15** with a valid window. No Antarctic/Himalayan/Greenland/NZ glacier in train (Dokriani dropped; Johnsons, Hurd, Mera, Pokalde routed to OOD). Headline model is 16/32 (~5,852 params). On this split 48/96 is better on seed-0 IID and pooled OOD.

**IID (seed 0).** PIKA 0.933 vs LSTM **0.800** vs PDD/TI 0.855 vs naive 1.403. LSTM wins every horizon, including h=1 = 0.490 vs PIKA 0.690 vs PDD/TI 0.498.

**IID (5 seeds).** PIKA 0.845 ± 0.046 vs LSTM 0.822 ± 0.033; PIKA wins 1/5; 95% CI for (LSTM − PIKA) = [−0.085, +0.041] (includes 0). Tie; LSTM’s mean is lower. Quote the 5-seed line, not seed 0.

**OOD (15 glaciers).** Large 48/96 **0.769**, blend 0.845, PIKA 16/32 0.901, LSTM 0.933, XGB 0.941, naive 0.949. 16/32 edges LSTM/naive on RMSE; 48/96 is the pooled winner. Do not claim OOD SOTA.

**LORO.** PIKA wins 1/10 regions; mean 0.80 vs LSTM 0.61; CI against PIKA. Do not claim transfer.

**WMAPE.** IID: PIKA 47.1% vs LSTM 40.9% vs PDD/TI 40.6% vs naive 73.9%. OOD: 48/96 55.3%, LSTM 62.0%, XGB 65.0%, PIKA 16/32 69.5%, naive 71.9%. On a fixed split WMAPE is a rescaling of MAE and cannot reorder models relative to MAE, but it does reorder them relative to RMSE: **PIKA 16/32's OOD edge over the LSTM reverses** (RMSE 0.901 vs 0.933 for us; WMAPE 69.5% vs 62.0% against us). Name the metric when stating the OOD ranking.

**Ablation (5 seeds, paired).** Full 0.8446 ± 0.0512. Paired deltas: no residual **+0.037** [−0.016,+0.074]; no quantile −0.001 [−0.035,+0.029]; 48/96 −0.008 [−0.035,+0.020]; no physics −0.005 [−0.018,+0.005]; no Lyapunov **exactly 0.000**. **Every CI contains zero — no component has a detectable IID effect.** The earlier +0.396 for residual learning came from an ablation that never retrained (it read a delta-trained head as an absolute balance, measuring a ~1.2 m w.e. offset); do not cite it. Lyapunov is provably inert: `relu(0.5 − max|eig|)` with eigenvalues pinned to [0.497, 0.95] is identically zero.

**UQ.** Raw 80% PI under-covers (~59%); conformal on a glacier split reaches ~84% at width 2.45 vs raw 1.28. Report both.


In [33]:
# ============================================================================
# Final Summary
# ============================================================================

print("\n" + "=" * 70)
print("  FINAL SUMMARY — PIKA for NeurIPS 2026 CCAI Workshop")
print("=" * 70)

pika_iid_m = compute_metrics(pikav2_iid, region_lookup)
lstm_iid_m = compute_metrics(lstm_iid, region_lookup)
naive_iid_m = compute_metrics(naive_iid, region_lookup)
pdd_iid_m = compute_metrics(pdd_iid, region_lookup)
pika_ood_m = compute_metrics(pikav2_ood, region_lookup)
lstm_ood_m = compute_metrics(lstm_ood, region_lookup)
naive_ood_m = compute_metrics(naive_ood, region_lookup)
xgb_ood_m = compute_metrics(xgb_ood, region_lookup)


def _h1_rmse(results):
    err = []
    for d in results.values():
        yp = np.asarray(d["y_pred"]).reshape(-1)
        yt = np.asarray(d["y_true"]).reshape(-1)
        err.append((yp[0] - yt[0]) ** 2)
    return float(np.sqrt(np.mean(err)))


n_params = sum(p.numel() for p in pikav2_model.parameters())
n_lstm = sum(p.numel() for p in lstm_model.parameters())

print(f"""
  Model: residual Koopman autoencoder (PIKA), {n_params:,} params
  LSTM:  2-layer, {n_lstm:,} params
  Task:  5-year glacier mass balance (HISTORY=5, FORECAST=5)

  PROTOCOL:
    - Train years <= 2018; IID = last strict window with fut >= 2019
    - OOD = 20 assigned holdouts (15 with a valid 2019+ window)
    - No Antarctic/Himalayan/Greenland/NZ glacier in training
    - Dokriani dropped; Johnsons/Hurd/Mera/Pokalde routed to OOD

  IID TEMPORAL (seed 0, {pika_iid_m['n_glaciers']} glaciers):
    PIKA:              {pika_iid_m['pooled_rmse']:.4f}  (h=1 {_h1_rmse(pikav2_iid):.4f})
    LSTM:                 {lstm_iid_m['pooled_rmse']:.4f}  (h=1 {_h1_rmse(lstm_iid):.4f})
    PDD/TI (per-glacier): {pdd_iid_m['pooled_rmse']:.4f}
    Naive persistence:    {naive_iid_m['pooled_rmse']:.4f}
    LSTM wins seed-0 IID (including h=1). PDD/TI is a per-glacier linear fit, not a shared model.
""")

print("  OOD HOLDOUT ({} glaciers):".format(pika_ood_m["n_glaciers"]))
print(f"    Naive persistence:    {naive_ood_m['pooled_rmse']:.4f}")
print(f"    XGBoost:              {xgb_ood_m['pooled_rmse']:.4f}")
print(f"    LSTM:                 {lstm_ood_m['pooled_rmse']:.4f}")
print(f"    PIKA (16/32):      {pika_ood_m['pooled_rmse']:.4f}")
if "reduced_ood" in globals():
    _large_ood = compute_metrics(reduced_ood, region_lookup)
    print(f"    PIKA (large 48/96): {_large_ood['pooled_rmse']:.4f}   <-- pooled winner")
print("    Do not claim OOD SOTA. 48/96 wins pooled; 16/32 only edges LSTM/naive.")

if "seed_results" in globals() and seed_results.get("PIKA"):
    pa = np.array(seed_results["PIKA"])
    la = np.array(seed_results["LSTM"])
    print("\n  5-SEED IID POOLED RMSE:")
    print(f"    PIKA  mean={pa.mean():.4f} +/- {pa.std():.4f}  {np.round(pa, 4).tolist()}")
    print(f"    LSTM  mean={la.mean():.4f} +/- {la.std():.4f}  {np.round(la, 4).tolist()}")
    print(f"    PIKA wins {int((pa < la).sum())}/{len(pa)} seeds")
    print("    95% CI (LSTM - PIKA) includes 0: statistically tied.")
    print("    Quote the 5-seed mean, not seed 0.")
else:
    print("\n  5-SEED: run the multi-seed cell; seed 0 is not the paper result.")

if "loro_df" in globals() and len(loro_df) > 0:
    n_win = int(loro_df["pika_wins"].sum())
    print("\n  LORO ({} regions):".format(len(loro_df)))
    print(f"    PIKA mean={loro_df['pika_rmse'].mean():.4f}  LSTM mean={loro_df['lstm_rmse'].mean():.4f}")
    print(f"    PIKA wins {n_win}/{len(loro_df)} folds. Do not claim transfer.")
else:
    print("\n  LORO: run the LORO cell.")

print("""
  ABLATION (IID pooled): residual learning is load-bearing (~+0.39 if removed).
    Physics / Lyapunov ~ 0 or slightly better without them. 48/96 and no-quantile
    beat full 16/32 on this seed-0 split; quote 16/32 as the compact model.

  UQ: raw 80% PI under-covers; conformal on a glacier split is the honest interval.

  PAPER SENTENCE:
    Small residual Koopman (~6k) is statistically tied with a 9x larger LSTM on
    IID (5-seed CI includes 0; LSTM mean lower). LSTM wins every horizon and LORO.
    16/32 edges LSTM/naive on 15-glacier OOD; 48/96 is better still. Conformal UQ
    is reported. Physics is a regularizer that does not reduce RMSE.
""")
print("=" * 70)



  FINAL SUMMARY — PIKA for NeurIPS 2026 CCAI Workshop

  Model: residual Koopman autoencoder (PIKA), 5,852 params
  LSTM:  2-layer, 51,265 params
  Task:  5-year glacier mass balance (HISTORY=5, FORECAST=5)

  PROTOCOL:
    - Train years <= 2018; IID = last strict window with fut >= 2019
    - OOD = 20 assigned holdouts (15 with a valid 2019+ window)
    - No Antarctic/Himalayan/Greenland/NZ glacier in training
    - Dokriani dropped; Johnsons/Hurd/Mera/Pokalde routed to OOD

  IID TEMPORAL (seed 0, 65 glaciers):
    PIKA:              0.9332  (h=1 0.6901)
    LSTM:                 0.8000  (h=1 0.4896)
    PDD/TI (per-glacier): 0.8546
    Naive persistence:    1.4031
    LSTM wins seed-0 IID (including h=1). PDD/TI is a per-glacier linear fit, not a shared model.

  OOD HOLDOUT (15 glaciers):
    Naive persistence:    0.9486
    XGBoost:              0.9408
    LSTM:                 0.9329
    PIKA (16/32):      0.9014
    PIKA (large 48/96): 0.7690   <-- pooled winner
    Do not clai